In [ ]:
# ========= NB3 CELL 0 (one-time): remove files made by the mistaken run in old Notebook 3 =========
import os, glob
TARGETS = [
    "../data/processed_v2/splits_v2.csv",
    "../data/processed_v2/prep_random_split_train.csv",
    "../data/processed_v2/prep_random_split_test.csv",
    "../data/processed_v2/prep_temporal_split_train.csv",
    "../data/processed_v2/prep_temporal_split_test.csv",
    "../models_v2/prep_random_split.joblib",
    "../models_v2/prep_temporal_split.joblib",
    "v2_prep.py",
] + glob.glob("__pycache__/v2_prep*.pyc")
for f in TARGETS:
    if os.path.exists(f):
        os.remove(f); print("deleted :", f)
    else:
        print("not found:", f)
print("\nprocessed_v2 now contains:")
print(sorted(os.listdir("../data/processed_v2")))

In [ ]:
# ========= NB3 CELL 1: load final table, leakage check, make the two splits =========
# Reads ONLY processed_v2/model_table_v2.csv. Writes ONLY processed_v2/splits_v2.csv.
# The split is made ONCE here and every later cell reads it from the file.
import pandas as pd, numpy as np, hashlib
from sklearn.model_selection import train_test_split

P2 = "../data/processed_v2/"
SEED = 42
T = pd.read_csv(P2 + "model_table_v2.csv")
N = len(T)

ID_COLS = ["stay_id", "subject_id", "hadm_id"]
LABEL = "mortality_28d"
SPLIT_TAGS = ["est_calendar_year", "temporal_set"]
FEATURES = [c for c in T.columns if c not in ID_COLS + [LABEL] + SPLIT_TAGS]

# ---------------- leakage + integrity checks ----------------
assert N == 2777 and T.stay_id.is_unique and T.subject_id.is_unique
assert set(T[LABEL].unique()) <= {0, 1} and T[LABEL].notna().all()
WORDS = ["mortality", "death", "expire", "discharge", "outtime", "subtype", "anchor",
         "temporal", "immortal", "hospice", "calendar", "year", "delirium", "cam_"]
TOK = ["dod", "los", "died", "dead", "label", "target", "outcome", "id"]
bad = [c for c in FEATURES if any(w in c.lower() for w in WORDS)
       or any(t in c.lower().split("_") for t in TOK)]
assert not bad, f"leaky or ID columns among features: {bad}"
assert len(FEATURES) == 391, f"expected 391 features, got {len(FEATURES)}"

# ---------------- split A: random, stratified on the outcome ----------------
tr_idx, te_idx = train_test_split(T.index, test_size=0.20, random_state=SEED,
                                  stratify=T[LABEL])
S = T[["stay_id", "subject_id", LABEL, "temporal_set", "est_calendar_year"]].copy()
S["random_split"] = "train"
S.loc[te_idx, "random_split"] = "test"

# ---------------- split B: calendar era (development <= 2017, validation > 2017) ----------------
S["temporal_split"] = np.where(S.temporal_set == "development_era", "train", "test")

# ---------------- checks ----------------
for col in ["random_split", "temporal_split"]:
    tr = set(S.subject_id[S[col] == "train"]); te = set(S.subject_id[S[col] == "test"])
    assert not (tr & te), f"{col}: same patient in train and test"
    assert len(tr) + len(te) == N
assert (S.est_calendar_year[S.temporal_split == "train"] <= 2017).all()
assert (S.est_calendar_year[S.temporal_split == "test"] > 2017).all()

S = S.drop(columns=[LABEL, "temporal_set"])
S.to_csv(P2 + "splits_v2.csv", index=False)

def fp(ids):                                   # fingerprint so any later change is detectable
    return hashlib.md5(",".join(map(str, sorted(ids))).encode()).hexdigest()[:12]

print("Saved processed_v2/splits_v2.csv", S.shape)
print("Features available to the models:", len(FEATURES))
print("\n=== SPLIT SIZES AND 28-DAY MORTALITY ===")
for col in ["random_split", "temporal_split"]:
    for part in ["train", "test"]:
        m = S[col] == part
        y = T.loc[m.values, LABEL]
        print(f"{col:15s} {part:5s}: n={m.sum():5d}  deaths={int(y.sum()):4d}  "
              f"mortality={y.mean():.1%}  fingerprint={fp(S.stay_id[m])}")
print("\nRandom split - cancer type share (%), train vs test:")
ct = pd.crosstab(T.cancer_type, S.random_split, normalize="columns").mul(100).round(1)
print(ct.to_string())
print("\nTemporal split - patients per estimated year:")
print(pd.crosstab(S.est_calendar_year, S.temporal_split).to_string())
print("\nLeakage check: PASS | no patient in both train and test: PASS")

In [ ]:
%%writefile v2_prep.py
# v2_prep.py - training-only preprocessing used by every Notebook 3/4 v2 cell.
# Everything that is LEARNED from data (20% rule, constant/duplicate columns, clipping
# limits, MICE, Stability-Index scaling, standard scaling) is fitted on the rows passed
# to Prep.fit() - always a TRAINING set or a training fold - and only APPLIED elsewhere.
# The outcome is never passed in.
import numpy as np, pandas as pd
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler

P2 = "../data/processed_v2/"
ID_COLS = ["stay_id", "subject_id", "hadm_id"]
LABEL = "mortality_28d"
SPLIT_TAGS = ["est_calendar_year", "temporal_set"]
CSI_VITALS = ["HR", "MAP", "RR", "SpO2"]
# Lactate is a Sepsis-3 criterion and a core ICU mortality marker. Its missingness (~20%)
# sits exactly on the 20% cut-off, so the rule would keep or drop it by chance of the split.
# Pre-specified clinical exception: kept if <= 30% missing in the training rows.
PROTECTED = ["Lactate_mean", "Lactate_min", "Lactate_max", "Lactate_first", "Lactate_last",
             "Lactate_ever_abn", "Lactate_ever_severe", "Lactate_last_state",
             "hypotension_x_lactate"]
PROTECTED_MAX_MISSING = 0.30
CANCER_LEVELS = ["Gastrointestinal", "Genitourinary", "Hematologic",
                 "Other/Unspecified", "Respiratory/Thoracic"]


def load_table():
    T = pd.read_csv(P2 + "model_table_v2.csv")
    S = pd.read_csv(P2 + "splits_v2.csv")
    assert (T.stay_id.values == S.stay_id.values).all(), "split file out of order"
    feats = [c for c in T.columns if c not in ID_COLS + [LABEL] + SPLIT_TAGS]
    return T, S, feats


def add_fixed_features(X):
    """Fixed-rule features (no fitting): maximum deviation beyond the normal band."""
    X = X.copy()
    # a count of measurements / observed hours that is missing means "none": 0, not unknown
    for c in [c for c in X.columns if c.endswith(("_n", "_obs_hours"))]:
        X[c] = X[c].fillna(0)
    X["Lactate_measured"] = (X["Lactate_n"] > 0).astype(float)
    th = pd.read_csv(P2 + "tpife_thresholds.csv").set_index("variable")
    for v in CSI_VITALS:
        ml, mh = th.loc[v, "mild_low"], th.loc[v, "mild_high"]
        lo, hi = X[f"{v}_min"], X[f"{v}_max"]
        d = pd.Series(0.0, index=X.index)
        if pd.notna(ml):
            d = np.maximum(d, ml - lo)
        if pd.notna(mh):
            d = np.maximum(d, hi - mh)
        d[lo.isna() | hi.isna()] = np.nan
        X[f"{v}_max_dev"] = d
    return X


def one_hot(X):
    X = X.copy()
    for lv in CANCER_LEVELS:
        X["cancer_" + lv.split("/")[0].lower()] = (X.cancer_type == lv).astype(float)
    return X.drop(columns="cancer_type")


class Prep:
    def __init__(self, seed=42, max_missing=0.20):
        self.seed, self.max_missing = seed, max_missing

    def fit(self, X):
        X = one_hot(X)
        miss = X.isna().mean()
        self.train_missing = miss
        limit = pd.Series(self.max_missing, index=miss.index)
        limit[[c for c in PROTECTED if c in limit.index]] = PROTECTED_MAX_MISSING
        self.dropped_missing = sorted(miss[miss > limit].index)
        X = X.drop(columns=self.dropped_missing)
        nun = X.nunique(dropna=True)
        self.dropped_constant = sorted(nun[nun <= 1].index)
        X = X.drop(columns=self.dropped_constant)
        dup = X.T.duplicated()
        self.dropped_duplicate = sorted(X.columns[dup.values])
        X = X.drop(columns=self.dropped_duplicate)
        self.cols = list(X.columns)
        self.binary = [c for c in self.cols if set(X[c].dropna().unique()) <= {0, 1}]
        self.cont = [c for c in self.cols if c not in self.binary]
        self.lo = X[self.cont].quantile(0.005)
        self.hi = X[self.cont].quantile(0.995)
        X[self.cont] = X[self.cont].clip(self.lo, self.hi, axis=1)
        self.vmin, self.vmax = X.min(), X.max()
        self.imp = IterativeImputer(estimator=BayesianRidge(), max_iter=10,
                                    n_nearest_features=50, random_state=self.seed,
                                    skip_complete=True)
        self.imp.fit(X)
        Xi = self._finish_impute(X)
        self.csi_parts = {v: [c for c in [f"{v}_transitions_per24h", f"{v}_episodes_per24h",
                                          f"{v}_frac_abn", f"{v}_max_dev",
                                          f"{v}_longest_episode_h", f"stab_{v}_arv",
                                          f"stab_{v}_cv"] if c in Xi.columns]
                          for v in CSI_VITALS}
        allp = [c for p in self.csi_parts.values() for c in p]
        self.csi_mu, self.csi_sd = Xi[allp].mean(), Xi[allp].std().replace(0, 1)
        Xi = self._add_csi(Xi)
        self.out_cols = list(Xi.columns)
        self.scaler = StandardScaler().fit(Xi)
        return self

    def _finish_impute(self, X):
        Xi = pd.DataFrame(self.imp.transform(X[self.cols]), columns=self.cols, index=X.index)
        Xi = Xi.clip(self.vmin, self.vmax, axis=1)           # stay inside training range
        Xi[self.binary] = Xi[self.binary].round()
        return Xi

    def _add_csi(self, Xi):
        Xi = Xi.copy()
        for v, parts in self.csi_parts.items():
            if parts:
                z = (Xi[parts] - self.csi_mu[parts]) / self.csi_sd[parts]
                Xi[f"csi_{v}"] = z.mean(axis=1)               # higher = LESS stable
        cs = [f"csi_{v}" for v in CSI_VITALS if f"csi_{v}" in Xi]
        Xi["csi_total"] = Xi[cs].mean(axis=1)
        return Xi

    def transform(self, X, scaled=False):
        X = one_hot(X)
        X = X[self.cols].copy()
        X[self.cont] = X[self.cont].clip(self.lo, self.hi, axis=1)
        Xi = self._add_csi(self._finish_impute(X))[self.out_cols]
        if scaled:
            Xi = pd.DataFrame(self.scaler.transform(Xi), columns=self.out_cols, index=Xi.index)
        return Xi

    def complete_case_mask(self, X):
        """Patients with every kept (<=20% missing) feature actually measured."""
        X = one_hot(X)
        return X[self.cols].notna().all(axis=1)

In [ ]:
# ========= NB3 CELL 2b: fit the preparation on TRAINING patients only, apply to test =========
import os, joblib, time, importlib
import pandas as pd, numpy as np
import v2_prep
importlib.reload(v2_prep)
from v2_prep import load_table, add_fixed_features, Prep, LABEL

P2 = "../data/processed_v2/"; M2 = "../models_v2/"
os.makedirs(M2, exist_ok=True)
T, S, FEATURES = load_table()
X_all = add_fixed_features(T[FEATURES])
y_all = T[LABEL].astype(int)

for arm in ["random_split", "temporal_split"]:
    t = time.time()
    tr = (S[arm] == "train").values
    prep = Prep(seed=42).fit(X_all[tr])                  # <- training rows ONLY, no outcome
    Xtr, Xte = prep.transform(X_all[tr]), prep.transform(X_all[~tr])
    assert list(Xtr.columns) == list(Xte.columns) == prep.out_cols
    assert not Xtr.isna().any().any() and not Xte.isna().any().any()
    for part, Xp, m in [("train", Xtr, tr), ("test", Xte, ~tr)]:
        out = pd.concat([T.loc[m, ["stay_id"]].reset_index(drop=True),
                         y_all[m].reset_index(drop=True), Xp.reset_index(drop=True)], axis=1)
        out.to_csv(P2 + f"prep_{arm}_{part}.csv", index=False)
    joblib.dump(prep, M2 + f"prep_{arm}.joblib")
    cc_tr = prep.complete_case_mask(X_all[tr]); cc_te = prep.complete_case_mask(X_all[~tr])

    print("=" * 70, f"\n{arm.upper()}  (fitted on {tr.sum()} training patients, "
          f"{(time.time() - t) / 60:.1f} min)\n" + "=" * 70)
    print("Dropped, >20% missing in TRAIN:", len(prep.dropped_missing))
    bl = prep.train_missing[(prep.train_missing > 0.15) & (prep.train_missing <= 0.25)]
    print("Borderline features (15-25% missing in TRAIN) -> kept / dropped:")
    for f, v in bl.sort_values().items():
        print(f"   {f:32s} {v:6.1%}  {'DROPPED' if f in prep.dropped_missing else 'kept'}")
    print("Lactate kept:", all(c in prep.out_cols for c in ["Lactate_max", "Lactate_measured"]))
    print("Dropped, constant in TRAIN:", prep.dropped_constant)
    print("Dropped, duplicate of another column in TRAIN:", prep.dropped_duplicate)
    print("Final model inputs:", len(prep.out_cols),
          f"(binary {len(prep.binary)}, continuous {len(prep.cont)}, + CSI 5)")
    print("Stability-Index parts per vital:",
          {v: len(p) for v, p in prep.csi_parts.items()})
    print("Complete-case patients: train", int(cc_tr.sum()), "| test", int(cc_te.sum()))
    print("CSI total, train: median {:.2f} IQR {:.2f} to {:.2f}".format(
        Xtr.csi_total.median(), Xtr.csi_total.quantile(.25), Xtr.csi_total.quantile(.75)))
    print("Saved:", f"prep_{arm}_train.csv", Xtr.shape, "|", f"prep_{arm}_test.csv", Xte.shape,
          "|", f"models_v2/prep_{arm}.joblib")

print("\nNo missing values after preparation: PASS | test used only for .transform(): PASS")

In [ ]:
%%writefile v2_select.py
# v2_select.py - SHAP-RFE feature selection (CatBoost + TreeSHAP), used on a TRAINING set
# or inside a training fold only. Rules fixed in advance:
#  - each round: 5-fold inner CV, CatBoost (balanced class weights), SHAP importance
#    measured on the inner validation parts, weakest 20% of features removed
#  - stop at 10 features
#  - chosen set = the SMALLEST set whose inner-CV AUC is within 0.005 of the best round
#  - STABILITY SELECTION: the whole SHAP-RFE is repeated with 5 different fold seeds;
#    final set = features chosen in >= 3 of the 5 runs (majority)
import numpy as np, pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

STEP, MIN_FEATURES, INNER_FOLDS, TOL = 0.20, 10, 5, 0.005
SEEDS, MIN_VOTES = [42, 43, 44, 45, 46], 3


def _catboost(seed):
    return CatBoostClassifier(iterations=300, depth=4, learning_rate=0.05,
                              auto_class_weights="Balanced", random_seed=seed,
                              verbose=0, thread_count=-1, allow_writing_files=False)


def shap_rfe(X, y, seed=42):
    X = X.reset_index(drop=True); y = pd.Series(y).reset_index(drop=True)
    feats, history = list(X.columns), []
    skf = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=seed)
    while True:
        aucs, imp = [], np.zeros(len(feats))
        for tr, va in skf.split(X, y):
            m = _catboost(seed).fit(X.loc[tr, feats], y[tr])
            aucs.append(roc_auc_score(y[va], m.predict_proba(X.loc[va, feats])[:, 1]))
            sv = m.get_feature_importance(Pool(X.loc[va, feats], y[va]), type="ShapValues")
            imp += np.abs(sv[:, :-1]).mean(axis=0)
        history.append({"n_features": len(feats), "inner_auc": float(np.mean(aucs)),
                        "features": list(feats),
                        "importance": dict(zip(feats, imp / INNER_FOLDS))})
        if len(feats) <= MIN_FEATURES:
            break
        keep = max(MIN_FEATURES, int(len(feats) * (1 - STEP)))
        feats = [feats[i] for i in np.argsort(-imp)[:keep]]
    best = max(h["inner_auc"] for h in history)
    chosen = min([h for h in history if h["inner_auc"] >= best - TOL],
                 key=lambda h: h["n_features"])
    return chosen["features"], chosen["importance"], history


def stability_select(X, y):
    """SHAP-RFE repeated over SEEDS; keep features chosen in >= MIN_VOTES runs."""
    votes, shap_sum, runs = {}, {}, []
    for s in SEEDS:
        feats, imp, hist = shap_rfe(X, y, seed=s)
        runs.append({"seed": s, "n_chosen": len(feats),
                     "best_inner_auc": max(h["inner_auc"] for h in hist)})
        for f in feats:
            votes[f] = votes.get(f, 0) + 1
            shap_sum[f] = shap_sum.get(f, 0) + imp[f]
    tab = pd.DataFrame({"feature": list(votes), "votes": list(votes.values()),
                        "mean_abs_shap": [shap_sum[f] / votes[f] for f in votes]})
    tab = tab.sort_values(["votes", "mean_abs_shap"], ascending=False).reset_index(drop=True)
    final = tab.feature[tab.votes >= MIN_VOTES].tolist()
    return final, tab, pd.DataFrame(runs)


def evaluate_set(X, y, feats, seed=2024):
    """Common yardstick for comparing feature pools: default CatBoost, 5x3 repeated CV,
    fold seed different from the selection seeds. TRAINING data only."""
    from sklearn.model_selection import RepeatedStratifiedKFold
    X = X.reset_index(drop=True); y = pd.Series(y).reset_index(drop=True)
    aucs = []
    for tr, va in RepeatedStratifiedKFold(n_splits=5, n_repeats=3,
                                          random_state=seed).split(X, y):
        m = _catboost(seed).fit(X.loc[tr, feats], y[tr])
        aucs.append(roc_auc_score(y[va], m.predict_proba(X.loc[va, feats])[:, 1]))
    return float(np.mean(aucs)), float(np.std(aucs))


# ---------------- measurement-process features (how often something was charted) ----------------
def is_measurement(f):
    return f.endswith(("_n", "_obs_hours")) or f in ("Lactate_measured",
                                                     "tpife_n_vitals_observed")


# ---------------- feature family (one definition used in NB3 and NB4) ----------------
TPIFE_SUFFIX = ("_frac_abn", "_frac_severe", "_auc_dev", "_transitions_per24h",
                "_episodes_per24h", "_longest_episode_h", "_first_episode_h", "_unresolved",
                "_deterioration", "_max_dev", "_ever_abn", "_ever_severe", "_last_state",
                "_late_minus_early_mean")
ORGAN_SUPPORT = {"vaso_any", "dobutamine_any", "n_vasoactive_agents", "nee_max",
                 "nee_mean_24h", "vaso_hours", "fio2_max", "pf_min", "sf_min",
                 "resp_support_level", "uo_ml_kg_h"}


def feature_family(f):
    if f.startswith("csi_"): return "Composite Stability Index"
    if f.startswith("osbi_"): return "organ burden (OSBI)"
    if f.startswith("stab_"): return "stability component"
    if f.startswith("tpife_"): return "TPIFE stage 4 (aggregation)"
    if f.startswith("elix_"): return "comorbidity (Elixhauser)"
    if f.startswith("cancer_") or f in ("age_at_admission", "sex_male"):
        return "demographics/cancer"
    if f.startswith(("sofa_", "ap_", "apache_")): return "severity score"
    if f in ("weight_kg", "gcs_verbal_ETT_24h"): return "context"
    if f in ORGAN_SUPPORT: return "organ support"
    if f.startswith(("sirs_", "hypotension_", "hypoxaemia_", "creat_rise", "shock_index")):
        return "interaction / SIRS"
    if f.endswith(TPIFE_SUFFIX) or any(s in f for s in ("_early_", "_mid_", "_late_")):
        return "TPIFE stages 1-3"
    return "basic temporal"


TPIFE_FAMILIES = {"TPIFE stages 1-3", "TPIFE stage 4 (aggregation)",
                  "Composite Stability Index", "stability component", "organ burden (OSBI)"}

In [ ]:
# ========= NB3 CELL 3b (v2): stability SHAP-RFE + pre-specified feature-pool decision =========
# TRAINING data only (prep_*_train.csv). Test files are never opened in this cell.
# Pool A = all inputs | Pool B = A without measurement-process features (how often charted)
# Pool C = B without TPIFE / CSI / stability / OSBI families (information only)
# Rule fixed in advance: use Pool B (physiology only, more transportable) if its AUC is
# within 0.005 of Pool A; otherwise keep Pool A.
import time, importlib, json
import pandas as pd, numpy as np
import v2_select
importlib.reload(v2_select)
from v2_select import (stability_select, evaluate_set, feature_family, is_measurement,
                       TPIFE_FAMILIES, SEEDS, MIN_VOTES)

P2, M2 = "../data/processed_v2/", "../models_v2/"
summary = {}
for arm in ["random_split", "temporal_split"]:
    t = time.time()
    tr = pd.read_csv(P2 + f"prep_{arm}_train.csv")
    y = tr.pop("mortality_28d").astype(int)
    X = tr.drop(columns="stay_id")
    poolA = list(X.columns)
    poolB = [f for f in poolA if not is_measurement(f)]
    poolC = [f for f in poolB if feature_family(f) not in TPIFE_FAMILIES]
    res = {}
    for name, pool in [("A_all", poolA), ("B_no_measurement", poolB), ("C_no_TPIFE", poolC)]:
        final, tab, runs = stability_select(X[pool], y)
        auc, sd = evaluate_set(X, y, final)
        res[name] = {"features": final, "n": len(final), "auc": auc, "sd": sd}
        tab["family"] = tab.feature.map(feature_family)
        tab.to_csv(P2 + f"selection_votes_{arm}_{name}.csv", index=False)
        print(f"{arm} | pool {name:17s}: {len(pool):3d} inputs -> {len(final):3d} stable features"
              f" | CV AUC {auc:.4f} (sd {sd:.4f}) | per-seed sizes {runs.n_chosen.tolist()}")

    use = "B_no_measurement" if res["B_no_measurement"]["auc"] >= res["A_all"]["auc"] - 0.005 \
        else "A_all"
    final = res[use]["features"]
    tab = pd.read_csv(P2 + f"selection_votes_{arm}_{use}.csv")
    sel = tab[tab.feature.isin(final)].copy()
    sel.to_csv(P2 + f"selected_features_{arm}.csv", index=False)
    summary[arm] = {k: {"n": v["n"], "auc": round(v["auc"], 4), "sd": round(v["sd"], 4)}
                    for k, v in res.items()}
    summary[arm]["pool_used"] = use

    print("=" * 70, f"\n{arm.upper()}  ({len(X)} training patients, "
          f"{(time.time() - t) / 60:.1f} min)\n" + "=" * 70)
    print(f"Rule: B used if AUC_B >= AUC_A - 0.005 -> pool used: {use}")
    print(f"TPIFE contribution (information): AUC {res['B_no_measurement']['auc']:.4f} with "
          f"vs {res['C_no_TPIFE']['auc']:.4f} without TPIFE families")
    print(f"Final features ({len(final)}), chosen in >= {MIN_VOTES} of {len(SEEDS)} runs:")
    print("By family:", sel.family.value_counts().to_dict())
    print(sel.to_string(index=False))
    print()

json.dump(summary, open(M2 + "feature_pool_decision.json", "w"), indent=2)
print("Saved: selected_features_<arm>.csv (final), selection_votes_*.csv, "
      "models_v2/feature_pool_decision.json")
print("Test data opened in this cell: NO")

In [ ]:
# ========= NB3 CELL 3c: ONE feature-pool decision for the whole study (no re-selection) =========
# The pool rule is a property of the PIPELINE, so it is decided once on the primary
# (random-split) development data and then applied unchanged to the era arm.
# Pool A (with measurement-count features) is kept as a pre-specified SENSITIVITY analysis.
# Reads only the selection files saved by cell 3b (training data). Test files not opened.
import json
import pandas as pd

P2, M2 = "../data/processed_v2/", "../models_v2/"
dec = json.load(open(M2 + "feature_pool_decision.json"))
PRIMARY_POOL = dec["random_split"]["pool_used"]          # decided on the primary arm
assert PRIMARY_POOL == "B_no_measurement", PRIMARY_POOL

for arm in ["random_split", "temporal_split"]:
    for pool, out in [(PRIMARY_POOL, f"selected_features_{arm}.csv"),
                      ("A_all", f"selected_features_{arm}_sensitivityA.csv"),
                      ("C_no_TPIFE", f"selected_features_{arm}_noTPIFE.csv")]:
        v = pd.read_csv(P2 + f"selection_votes_{arm}_{pool}.csv")
        v[v.votes >= 3].to_csv(P2 + out, index=False)
    sel = pd.read_csv(P2 + f"selected_features_{arm}.csv")
    assert not sel.feature.str.endswith(("_n", "_obs_hours")).any()
    dec[arm]["pool_used"] = PRIMARY_POOL
    dec[arm]["n_primary"] = len(sel)
    print(f"{arm:15s}: primary pool {PRIMARY_POOL} -> {len(sel)} features | "
          f"families {sel.family.value_counts().to_dict()}")

dec["rule"] = ("Pool decided once on the primary random-split training data "
               "(B if AUC_B >= AUC_A - 0.005) and applied to both arms; Pool A and "
               "no-TPIFE pool kept as pre-specified sensitivity/ablation analyses")
json.dump(dec, open(M2 + "feature_pool_decision.json", "w"), indent=2)
print("\nSaved: selected_features_<arm>.csv (primary), *_sensitivityA.csv, *_noTPIFE.csv")
print("Measurement-count features in primary sets: NONE (checked)")
print("Test data opened in this cell: NO")

In [ ]:
%%writefile v2_models.py
# v2_models.py - model definitions + Optuna search spaces (one copy used by NB3 and NB4).
# Class imbalance: class weights in every model (no resampling) unless cell 4 shows
# SMOTEENN is better by > 0.005 CV AUC (rule fixed in advance).
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

MODELS = ["LR", "DT", "RF", "XGB", "CatBoost", "LightGBM"]


# Search ranges. Widened after the first run, where XGB/LightGBM depth, XGB min_child_weight
# and RF depth hit the edge of the range (an optimum at the edge may lie outside it).
# kind: "int" | "float" | "logfloat" | "cat"
SPACES = {
    "LR": {"C": ("logfloat", 1e-4, 100)},
    "DT": {"max_depth": ("int", 2, 12), "min_samples_leaf": ("int", 5, 150)},
    "RF": {"n_estimators": ("int", 300, 1500, 100), "max_depth": ("int", 3, 30),
           "min_samples_leaf": ("int", 1, 30),
           "max_features": ("cat", ["sqrt", "log2", 0.3, 0.5])},
    "XGB": {"n_estimators": ("int", 200, 2000, 100), "learning_rate": ("logfloat", 0.003, 0.1),
            "max_depth": ("int", 1, 6), "subsample": ("float", 0.4, 1.0),
            "colsample_bytree": ("float", 0.3, 1.0), "min_child_weight": ("int", 1, 60),
            "reg_lambda": ("logfloat", 0.01, 30)},
    "CatBoost": {"iterations": ("int", 300, 2000, 100),
                 "learning_rate": ("logfloat", 0.003, 0.1),
                 "depth": ("int", 2, 8), "l2_leaf_reg": ("logfloat", 0.5, 30),
                 "bagging_temperature": ("float", 0, 1)},
    "LightGBM": {"n_estimators": ("int", 200, 2000, 100),
                 "learning_rate": ("logfloat", 0.003, 0.1),
                 "num_leaves": ("int", 3, 63), "max_depth": ("int", 1, 8),
                 "subsample": ("float", 0.4, 1.0), "colsample_bytree": ("float", 0.3, 1.0),
                 "min_child_samples": ("int", 5, 200)},
}


def space(name, t):
    p = {}
    for k, spec in SPACES[name].items():
        kind = spec[0]
        if kind == "int":
            step = spec[3] if len(spec) > 3 else 1
            p[k] = t.suggest_int(k, spec[1], spec[2], step=step)
        elif kind == "float":
            p[k] = t.suggest_float(k, spec[1], spec[2])
        elif kind == "logfloat":
            p[k] = t.suggest_float(k, spec[1], spec[2], log=True)
        else:
            p[k] = t.suggest_categorical(k, spec[1])
    return p


def at_boundary(name, best):
    """Numeric parameters whose best value sits on the edge of the search range."""
    out = []
    for k, spec in SPACES[name].items():
        if spec[0] == "cat":
            continue
        lo, hi = spec[1], spec[2]
        v = best[k]
        if spec[0] == "logfloat":
            near = v <= lo * 1.05 or v >= hi / 1.05
        else:
            step = spec[3] if len(spec) > 3 else (1 if spec[0] == "int" else (hi - lo) * 0.02)
            near = v <= lo + step * 0.5 or v >= hi - step * 0.5
        if near:
            out.append(f"{k}={v}")
    return out


def build(name, p, y_train=None, seed=42, weighted=True):
    """y_train is only used to count classes for XGBoost's weight (no outcome leakage:
    it is the training fold's own labels)."""
    if name == "LR":
        return make_pipeline(StandardScaler(), LogisticRegression(
            C=p["C"], class_weight="balanced" if weighted else None, max_iter=5000))
    if name == "DT":
        return DecisionTreeClassifier(class_weight="balanced" if weighted else None,
                                      random_state=seed, **p)
    if name == "RF":
        return RandomForestClassifier(class_weight="balanced_subsample" if weighted else None,
                                      n_jobs=-1, random_state=seed, **p)
    if name == "XGB":
        spw = float((y_train == 0).sum() / (y_train == 1).sum()) if weighted else 1.0
        return XGBClassifier(scale_pos_weight=spw, eval_metric="logloss", n_jobs=-1,
                             random_state=seed, tree_method="hist", **p)
    if name == "CatBoost":
        return CatBoostClassifier(auto_class_weights="Balanced" if weighted else None,
                                  random_seed=seed, verbose=0, thread_count=-1,
                                  allow_writing_files=False, **p)
    if name == "LightGBM":
        return LGBMClassifier(class_weight="balanced" if weighted else None, subsample_freq=1,
                              random_state=seed, n_jobs=-1, verbose=-1, **p)
    raise ValueError(name)

In [ ]:
# ========= NB3 CELL 4b: Optuna tuning + class-weights vs SMOTEENN (TRAINING data only) =========
# Uses prep_*_train.csv + selected_features_*.csv. Test files are not opened.
# These CV numbers are for CHOOSING settings only - the reported CV comes from cell 5.
import json, time, importlib, warnings, os
import numpy as np, pandas as pd, optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTEENN
import v2_models
importlib.reload(v2_models)
from v2_models import MODELS, space, build, at_boundary

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
P2, M2 = "../data/processed_v2/", "../models_v2/"
N_TRIALS = 80                      # tree ensembles; LR and DT use 40 (fixed in advance)
SEED = 42

for arm in ["random_split", "temporal_split"]:
    tr = pd.read_csv(P2 + f"prep_{arm}_train.csv")
    y = tr.pop("mortality_28d").astype(int).values
    feats = pd.read_csv(P2 + f"selected_features_{arm}.csv").feature.tolist()
    X = tr[feats]
    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
    print("=" * 70, f"\n{arm.upper()}  ({len(X)} patients, {len(feats)} features)\n" + "=" * 70)

    best, rows = {}, []
    for name in MODELS:
        t = time.time()
        def objective(trial):
            m = build(name, space(name, trial), y_train=y, seed=SEED)
            return cross_val_score(m, X, y, cv=skf, scoring="roc_auc", n_jobs=1).mean()
        study = optuna.create_study(direction="maximize",
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(objective, n_trials=N_TRIALS if name not in ["LR", "DT"] else 40)
        best[name] = study.best_params
        study.trials_dataframe().to_csv(M2 + f"optuna_{arm}_{name}.csv", index=False)
        edge = at_boundary(name, study.best_params)
        rows.append({"model": name, "tuning_cv_auc": round(study.best_value, 4),
                     "minutes": round((time.time() - t) / 60, 1), "at_edge": ";".join(edge)})
        print(f"{name:9s} best tuning CV AUC {study.best_value:.4f} "
              f"({(time.time() - t) / 60:.1f} min)  {study.best_params}")
        print(f"          parameters at the edge of the range: {edge if edge else 'none'}")

    # ---- imbalance handling: class weights vs SMOTEENN, same folds, tuned XGB + LightGBM ----
    print("\nClass weights vs SMOTEENN (same 5 folds, tuned settings):")
    imb = {}
    for name in ["XGB", "LightGBM"]:
        cw = cross_val_score(build(name, best[name], y_train=y, seed=SEED), X, y,
                             cv=skf, scoring="roc_auc").mean()
        sm = cross_val_score(ImbPipeline([("smoteenn", SMOTEENN(random_state=SEED)),
                                          ("m", build(name, best[name], y_train=y, seed=SEED,
                                                      weighted=False))]),
                             X, y, cv=skf, scoring="roc_auc").mean()
        imb[name] = {"class_weights": round(cw, 4), "smoteenn": round(sm, 4)}
        print(f"   {name:9s} class weights {cw:.4f} | SMOTEENN {sm:.4f}")
    gain = np.mean([v["smoteenn"] - v["class_weights"] for v in imb.values()])
    method = "smoteenn" if gain > 0.005 else "class_weights"
    print(f"Mean SMOTEENN gain {gain:+.4f} -> rule (> +0.005 needed): use {method.upper()}")

    json.dump({"best_params": best, "imbalance_method": method, "imbalance_check": imb,
               "n_trials": N_TRIALS, "seed": SEED, "features": feats},
              open(M2 + f"tuning_{arm}.json", "w"), indent=2)
    pd.DataFrame(rows).to_csv(M2 + f"tuning_summary_{arm}.csv", index=False)
    print("Saved:", f"models_v2/tuning_{arm}.json")

print("\nTest data opened in this cell: NO")

In [ ]:
# ========= NB3 CELL 5: NESTED repeated 5x5 cross-validation (random-split TRAINING set) =========
# Every outer fold repeats the WHOLE pipeline on its own training part only:
#   preparation (20% rule, MICE, clipping, CSI)  ->  stability SHAP-RFE (3 pools)
#   ->  all models with the cell-4 tuned settings  ->  predict the held-out fold.
# Test patients (random_split == "test") are never loaded into this cell.
# Progress is saved after every fold; if the run stops, just run the cell again.
import os, json, time, importlib, warnings
import numpy as np, pandas as pd, xgboost as xgb
from catboost import Pool
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss
import v2_prep, v2_select, v2_models
for m in (v2_prep, v2_select, v2_models):
    importlib.reload(m)
from v2_prep import load_table, add_fixed_features, Prep, LABEL
from v2_select import stability_select, is_measurement, feature_family, TPIFE_FAMILIES
from v2_models import build

warnings.filterwarnings("ignore")
P2, M2 = "../data/processed_v2/", "../models_v2/"
OUT = P2 + "cv/"
os.makedirs(OUT, exist_ok=True)
N_SPLITS, N_REPEATS, CV_SEED = 5, 5, 2025            # fixed in advance
TREE_MODELS = ["RF", "XGB", "CatBoost", "LightGBM"]
ALL_MODELS = ["LR", "DT"] + TREE_MODELS
SOFT_VOTE = ["XGB", "RF", "CatBoost"]                 # roadmap definition
STACK_BASE = ["LR", "RF", "XGB", "CatBoost", "LightGBM"]

T, S, FEATURES = load_table()
dev = (S.random_split == "train").values               # training patients ONLY
X_raw = add_fixed_features(T.loc[dev, FEATURES]).reset_index(drop=True)
y_all = T.loc[dev, LABEL].astype(int).reset_index(drop=True)
ids = T.loc[dev, "stay_id"].reset_index(drop=True)
params = json.load(open(M2 + "tuning_random_split.json"))["best_params"]
print(f"Nested CV on {len(X_raw)} training patients | {N_REPEATS}x{N_SPLITS} folds")


def predict_all(Xtr, ytr, Xva, feats, with_stacking):
    out, fitted = {}, {}
    for name in ALL_MODELS:
        m = build(name, params[name], y_train=ytr.values, seed=42).fit(Xtr[feats], ytr)
        out[name] = m.predict_proba(Xva[feats])[:, 1]
        fitted[name] = m
    out["SoftVoting"] = np.mean([out[n] for n in SOFT_VOTE], axis=0)
    if with_stacking:
        st = StackingClassifier(
            estimators=[(n, build(n, params[n], y_train=ytr.values, seed=42)) for n in STACK_BASE],
            final_estimator=LogisticRegression(max_iter=1000), stack_method="predict_proba",
            cv=StratifiedKFold(5, shuffle=True, random_state=42), n_jobs=1)
        out["Stacking"] = st.fit(Xtr[feats], ytr).predict_proba(Xva[feats])[:, 1]
    return out, fitted


def shap_means(fitted, Xva, feats):
    rows = []
    sv = {"XGB": fitted["XGB"].get_booster().predict(xgb.DMatrix(Xva[feats]),
                                                       pred_contribs=True)[:, :-1],
          "LightGBM": fitted["LightGBM"].predict(Xva[feats], pred_contrib=True)[:, :-1],
          "CatBoost": fitted["CatBoost"].get_feature_importance(
              Pool(Xva[feats]), type="ShapValues")[:, :-1]}
    for name, v in sv.items():
        for f, val in zip(feats, np.abs(v).mean(axis=0)):
            rows.append({"model": name, "feature": f, "mean_abs_shap": float(val)})
    return rows


done = set()
if os.path.exists(OUT + "cv_fold_metrics.csv"):
    fm = pd.read_csv(OUT + "cv_fold_metrics.csv")
    done = set(zip(fm.repeat, fm.fold))
    print(f"Resuming: {len(done)} folds already finished")

rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=CV_SEED)
for k, (tr, va) in enumerate(rskf.split(X_raw, y_all)):
    rep, fold = k // N_SPLITS + 1, k % N_SPLITS + 1
    if (rep, fold) in done:
        continue
    t = time.time()
    prep = Prep(seed=42).fit(X_raw.iloc[tr])                      # fold-training rows only
    Xtr, Xva = prep.transform(X_raw.iloc[tr]), prep.transform(X_raw.iloc[va])
    ytr, yva = y_all.iloc[tr].reset_index(drop=True), y_all.iloc[va].values
    Xtr, Xva = Xtr.reset_index(drop=True), Xva.reset_index(drop=True)

    poolA = list(Xtr.columns)
    poolB = [f for f in poolA if not is_measurement(f)]
    poolC = [f for f in poolB if feature_family(f) not in TPIFE_FAMILIES]
    preds, sel_rows, shap_rows, met_rows = [], [], [], []
    for pool, cols in [("B_primary", poolB), ("A_with_measurement", poolA),
                       ("C_no_TPIFE", poolC)]:
        feats = stability_select(Xtr[cols], ytr)[0]
        sel_rows += [{"repeat": rep, "fold": fold, "pool": pool, "feature": f} for f in feats]
        out, fitted = predict_all(Xtr, ytr, Xva, feats, with_stacking=(pool == "B_primary"))
        if pool == "B_primary":
            for r in shap_means(fitted, Xva, feats):
                shap_rows.append({"repeat": rep, "fold": fold, **r})
            for score in ["sofa_total", "apache_ii_total"]:
                b = LogisticRegression(max_iter=1000).fit(Xtr[[score]], ytr)
                out["SOFA" if score == "sofa_total" else "APACHE-II"] = \
                    b.predict_proba(Xva[[score]])[:, 1]
        for name, p in out.items():
            met_rows.append({"repeat": rep, "fold": fold, "pool": pool, "model": name,
                             "auc": roc_auc_score(yva, p), "brier": brier_score_loss(yva, p),
                             "n_features": len(feats) if name not in ("SOFA", "APACHE-II")
                             else 1})
            preds.append(pd.DataFrame({"repeat": rep, "fold": fold, "pool": pool,
                                       "model": name, "stay_id": ids.iloc[va].values,
                                       "y": yva, "prob": p}))
    for fname, rows in [("cv_oof_predictions.csv", pd.concat(preds)),
                        ("cv_selected_features.csv", pd.DataFrame(sel_rows)),
                        ("cv_shap.csv", pd.DataFrame(shap_rows)),
                        ("cv_fold_metrics.csv", pd.DataFrame(met_rows))]:
        rows.to_csv(OUT + fname, mode="a", index=False, header=not os.path.exists(OUT + fname))
    b = pd.DataFrame(met_rows).query("pool == 'B_primary'").set_index("model").auc
    print(f"fold {rep}.{fold} done ({(time.time() - t) / 60:.1f} min) | primary AUC: "
          + ", ".join(f"{m} {b[m]:.3f}" for m in ["LR", "XGB", "SoftVoting", "Stacking",
                                                  "SOFA"]))

# ---------------- summary over all 25 folds ----------------
fm = pd.read_csv(OUT + "cv_fold_metrics.csv")
assert len(fm[["repeat", "fold"]].drop_duplicates()) == N_SPLITS * N_REPEATS, "folds missing"
summ = fm.groupby(["pool", "model"]).agg(auc_mean=("auc", "mean"), auc_sd=("auc", "std"),
                                         brier_mean=("brier", "mean"),
                                         n_feat=("n_features", "median")).round(4)
summ.to_csv(OUT + "cv_summary.csv")
print(f"\n=== NESTED REPEATED CV (mean over {N_SPLITS * N_REPEATS} folds) - PRIMARY POOL ===")
print(summ.loc["B_primary"].sort_values("auc_mean", ascending=False).to_string())

print("\n=== PRE-SPECIFIED ABLATIONS (paired by fold) ===")
piv = fm.pivot_table(index=["repeat", "fold", "model"], columns="pool", values="auc")
for other, label in [("C_no_TPIFE", "TPIFE contribution (primary - no TPIFE)"),
                     ("A_with_measurement", "measurement features (with - primary)")]:
    d = (piv["B_primary"] - piv[other]) if other == "C_no_TPIFE" \
        else (piv[other] - piv["B_primary"])
    g = d.groupby(level="model").agg(["mean", "std", lambda s: (s > 0).mean()])
    g.columns = ["mean_diff", "sd", "share_folds_positive"]
    g = g.dropna(subset=["mean_diff"])
    print(f"\n{label}:")
    print(g.round(4).to_string())

sf = pd.read_csv(OUT + "cv_selected_features.csv").query("pool == 'B_primary'")
freq = (sf.groupby("feature").size() / (N_SPLITS * N_REPEATS)).sort_values(ascending=False)
freq.rename("selection_frequency").to_csv(OUT + "cv_feature_selection_frequency.csv")
print(f"\nFeatures selected in >= 80% of the {N_SPLITS * N_REPEATS} folds: "
      f"{int((freq >= 0.8).sum())}")
print(freq.head(30).round(2).to_string())
print("\nTest data opened in this cell: NO")

In [ ]:
%%writefile v2_final.py
# v2_final.py - final-model helpers used by NB3 (cells 6-7) and NB4.
#  - soft_voting(): the roadmap Soft-Voting ensemble (XGB + RF + CatBoost, equal weights)
#  - CVAP: Cross Venn-Abers predictors (Vovk et al. 2015), written out explicitly because the
#    'venn-abers' package does not install on Python 3.13. Per fold an inductive Venn-Abers
#    predictor gives (p0, p1); folds are merged with the GEOMETRIC-mean formula of Vovk 2015:
#    p = GM(p1) / (GM(1 - p0) + GM(p1)); uncertainty = mean interval width (p1 - p0).
#  - calibration_slope_intercept(): logistic recalibration of predictions (Cox 1958).
import numpy as np
from sklearn.base import clone
from sklearn.ensemble import VotingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from v2_models import build

SOFT_VOTE = ["XGB", "RF", "CatBoost"]


def soft_voting(params, y_train, seed=42):
    return VotingClassifier([(n, build(n, params[n], y_train=y_train, seed=seed))
                             for n in SOFT_VOTE], voting="soft", n_jobs=1)


def _ivap(cal_s, cal_y, test_s):
    p0, p1 = np.empty(len(test_s)), np.empty(len(test_s))
    for i, s in enumerate(test_s):
        for lab, out in ((0, p0), (1, p1)):
            ir = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip")
            ir.fit(np.r_[cal_s, s], np.r_[cal_y, lab])
            out[i] = ir.predict([s])[0]
    return p0, p1


class CVAP:
    def __init__(self, base, n_splits=5, seed=42):
        self.base, self.n_splits, self.seed = base, n_splits, seed

    def fit(self, X, y):
        X = X.reset_index(drop=True); y = np.asarray(y)
        self.parts_ = []
        for tr, ca in StratifiedKFold(self.n_splits, shuffle=True,
                                      random_state=self.seed).split(X, y):
            m = clone(self.base).fit(X.iloc[tr], y[tr])
            self.parts_.append((m, m.predict_proba(X.iloc[ca])[:, 1], y[ca]))
        return self

    def predict(self, X):
        P0, P1 = [], []
        for m, cal_s, cal_y in self.parts_:
            p0, p1 = _ivap(cal_s, cal_y, m.predict_proba(X)[:, 1])
            P0.append(p0); P1.append(p1)
        P0, P1 = np.clip(np.array(P0), 1e-6, 1 - 1e-6), np.clip(np.array(P1), 1e-6, 1 - 1e-6)
        gm = lambda a: np.exp(np.log(a).mean(axis=0))
        p = gm(P1) / (gm(1 - P0) + gm(P1))
        return {"p": p, "p0": P0.mean(axis=0), "p1": P1.mean(axis=0),
                "width": (P1 - P0).mean(axis=0)}


def calibration_slope_intercept(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    lp = np.log(p / (1 - p)).reshape(-1, 1)
    slope = LogisticRegression(C=1e6, max_iter=1000).fit(lp, y).coef_[0][0]
    # calibration-in-the-large: intercept with the linear predictor as an offset
    from scipy.optimize import minimize_scalar
    nll = lambda a: -np.mean(y * (a + lp[:, 0]) - np.log1p(np.exp(a + lp[:, 0])))
    intercept = minimize_scalar(nll, bounds=(-5, 5), method="bounded").x
    return float(slope), float(intercept)


def youden_threshold(y, p):
    from sklearn.metrics import roc_curve
    fpr, tpr, thr = roc_curve(y, p)
    j = np.argmax(tpr - fpr)
    return float(thr[j]), float(tpr[j]), float(1 - fpr[j])

In [ ]:
# ========= NB3 CELL 6b: final model choice (written rule) + fit on TRAINING data =========
# Uses training files only. Test files are not opened. Saves one bundle per arm for cell 7.
import json, time, importlib, warnings
import numpy as np, pandas as pd, joblib
from scipy import stats
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, brier_score_loss
import v2_prep, v2_select, v2_models, v2_final
for m in (v2_prep, v2_select, v2_models, v2_final):
    importlib.reload(m)
from v2_prep import load_table, add_fixed_features, LABEL
from v2_select import feature_family, is_measurement, TPIFE_FAMILIES
from v2_models import build
from v2_final import soft_voting, CVAP, calibration_slope_intercept, youden_threshold

warnings.filterwarnings("ignore")
P2, M2 = "../data/processed_v2/", "../models_v2/"
SEED = 42

# ---------------- 1. apply the pre-written selection rule to the nested-CV results -------
cv = pd.read_csv(P2 + "cv/cv_summary.csv").query("pool == 'B_primary'").set_index("model")
CAND = ["LR", "RF", "XGB", "CatBoost", "LightGBM", "SoftVoting", "Stacking"]
RANK = {"LR": 0, "RF": 1, "XGB": 1, "CatBoost": 1, "LightGBM": 1, "SoftVoting": 2, "Stacking": 3}
c = cv.loc[CAND]
best_auc = c.auc_mean.max()
near = c[c.auc_mean >= best_auc - 0.005].copy()
near["rank"] = near.index.map(RANK)
near = near.sort_values(["rank", "auc_mean", "brier_mean"], ascending=[True, False, True])
FINAL = near.index[0]
RUNNER_UP = c.drop(FINAL).auc_mean.idxmax()
print("Nested-CV AUC (primary pool):", c.auc_mean.round(4).to_dict())
print(f"Best {c.auc_mean.idxmax()} {best_auc:.4f}; within 0.005: {list(near.index)}")
print(f"FINAL MODEL by the rule: {FINAL} | CV runner-up for DeLong: {RUNNER_UP}")
assert FINAL == "SoftVoting", "rule picked a different model - send me this output"

# ---------------- 2. corrected resampled t-tests on the 25 folds (Nadeau & Bengio 2003) ----------
fm = pd.read_csv(P2 + "cv/cv_fold_metrics.csv")
piv = fm.pivot_table(index=["repeat", "fold"], columns=["pool", "model"], values="auc")
k_r = len(piv); ratio = 1 / 4                          # n_test / n_train in 5-fold CV
def corrected_t(d):
    t = d.mean() / np.sqrt((1 / k_r + ratio) * d.var(ddof=1))
    return d.mean(), 2 * stats.t.sf(abs(t), df=k_r - 1)
tests = {"SoftVoting: primary vs no-TPIFE": piv[("B_primary", "SoftVoting")]
         - piv[("C_no_TPIFE", "SoftVoting")],
         "SoftVoting: with measurement vs primary": piv[("A_with_measurement", "SoftVoting")]
         - piv[("B_primary", "SoftVoting")],
         "SoftVoting vs Stacking": piv[("B_primary", "SoftVoting")]
         - piv[("B_primary", "Stacking")],
         "SoftVoting vs SOFA": piv[("B_primary", "SoftVoting")] - piv[("B_primary", "SOFA")],
         "SoftVoting vs APACHE-II": piv[("B_primary", "SoftVoting")]
         - piv[("B_primary", "APACHE-II")]}
rows = []
print("\nCorrected resampled t-tests (25 folds, CV only):")
for name, d in tests.items():
    md, p = corrected_t(d)
    rows.append({"comparison": name, "mean_auc_diff": round(md, 4), "p_value": round(p, 4)})
    print(f"   {name:42s} diff {md:+.4f}  p = {p:.4f}")
pd.DataFrame(rows).to_csv(P2 + "cv/cv_corrected_ttests.csv", index=False)

# ---------------- 3. fit final + comparator models per arm (TRAINING data only) ----------------
T, S, FEATURES = load_table()
X_raw_all = add_fixed_features(T[FEATURES])
for arm in ["random_split", "temporal_split"]:
    t0 = time.time()
    tr = pd.read_csv(P2 + f"prep_{arm}_train.csv")
    y = tr.pop("mortality_28d").astype(int).values
    sid = tr.pop("stay_id").values
    params = json.load(open(M2 + f"tuning_{arm}.json"))["best_params"]
    feats = pd.read_csv(P2 + f"selected_features_{arm}.csv").feature.tolist()
    fA = pd.read_csv(P2 + f"selected_features_{arm}_sensitivityA.csv").feature.tolist()
    fC = pd.read_csv(P2 + f"selected_features_{arm}_noTPIFE.csv").feature.tolist()
    X = tr
    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
    B = {}

    # (a) final model: Soft-Voting + isotonic calibration (cv=5, one final refit)
    base = soft_voting(params, y, SEED)
    B["final_calibrated"] = CalibratedClassifierCV(base, method="isotonic", cv=5,
                                                   ensemble=False).fit(X[feats], y)
    B["final_uncalibrated"] = soft_voting(params, y, SEED).fit(X[feats], y)
    # (b) honest out-of-fold calibrated predictions -> threshold (Youden) + internal checks
    oof = cross_val_predict(CalibratedClassifierCV(soft_voting(params, y, SEED),
                                                   method="isotonic", cv=5, ensemble=False),
                            X[feats], y, cv=skf, method="predict_proba")[:, 1]
    thr, sens, spec = youden_threshold(y, oof)
    slope, inter = calibration_slope_intercept(y, oof)
    # (c) uncertainty: Cross Venn-Abers on the same Soft-Voting model
    B["cvap"] = CVAP(soft_voting(params, y, SEED), n_splits=5, seed=SEED).fit(X[feats], y)
    # (d) comparators for DeLong + ablations (uncalibrated; AUC compares ranking only)
    B["Stacking"] = StackingClassifier(
        estimators=[(n, build(n, params[n], y_train=y, seed=SEED))
                    for n in ["LR", "RF", "XGB", "CatBoost", "LightGBM"]],
        final_estimator=LogisticRegression(max_iter=1000), stack_method="predict_proba",
        cv=skf, n_jobs=1).fit(X[feats], y)
    for name in ["LR", "RF", "XGB", "CatBoost", "LightGBM", "DT"]:
        B[name] = build(name, params[name], y_train=y, seed=SEED).fit(X[feats], y)
    B["SOFA"] = LogisticRegression(max_iter=1000).fit(X[["sofa_total"]], y)
    B["APACHE-II"] = LogisticRegression(max_iter=1000).fit(X[["apache_ii_total"]], y)
    B["SV_sensitivityA"] = soft_voting(params, y, SEED).fit(X[fA], y)
    B["SV_noTPIFE"] = soft_voting(params, y, SEED).fit(X[fC], y)
    # (e) complete-case refit: only patients with every kept feature truly measured
    prep = joblib.load(M2 + f"prep_{arm}.joblib")
    raw_tr = X_raw_all.set_index(T.stay_id).loc[sid]
    cc = prep.complete_case_mask(raw_tr).values
    B["SV_complete_case"] = soft_voting(params, y[cc], SEED).fit(X.loc[cc, feats], y[cc])
    # (f) Multi-View diagnostic: one XGB per view on ALL primary-pool features of that view,
    #     meta-learner (LR) trained on out-of-fold view predictions only
    pool = [f for f in X.columns if not is_measurement(f)]
    VIEWS = {"Static": [f for f in pool if feature_family(f) in
                        {"demographics/cancer", "comorbidity (Elixhauser)", "severity score",
                         "context"}],
             "Physiology": [f for f in pool if feature_family(f) in
                            {"basic temporal", "organ support"}],
             "TPIFE": [f for f in pool if feature_family(f) in TPIFE_FAMILIES],
             "Interaction": [f for f in pool if feature_family(f) == "interaction / SIRS"]}
    view_oof = {}
    for v, cols in VIEWS.items():
        mdl = build("XGB", params["XGB"], y_train=y, seed=SEED)
        view_oof[v] = cross_val_predict(mdl, X[cols], y, cv=skf, method="predict_proba")[:, 1]
        B[f"view_{v}"] = build("XGB", params["XGB"], y_train=y, seed=SEED).fit(X[cols], y)
    meta_X = np.column_stack([view_oof[v] for v in VIEWS])
    B["view_meta"] = LogisticRegression(max_iter=1000).fit(meta_X, y)
    meta_oof = cross_val_predict(LogisticRegression(max_iter=1000), meta_X, y, cv=skf,
                                 method="predict_proba")[:, 1]

    B.update({"features": feats, "features_sensitivityA": fA, "features_noTPIFE": fC,
              "views": VIEWS, "threshold": thr, "final_model_type": FINAL,
              "train_complete_case_n": int(cc.sum()), "params": params})
    joblib.dump(B, M2 + f"final_{arm}.joblib")
    pd.DataFrame({"stay_id": sid, "y": y, "oof_calibrated": oof}).to_csv(
        P2 + f"train_oof_final_{arm}.csv", index=False)

    print("=" * 70, f"\n{arm.upper()} ({len(y)} training patients, {len(feats)} features, "
          f"{(time.time() - t0) / 60:.1f} min)\n" + "=" * 70)
    print(f"Out-of-fold (training) calibrated Soft-Voting: AUC {roc_auc_score(y, oof):.4f} | "
          f"Brier {brier_score_loss(y, oof):.4f} | slope {slope:.3f} | intercept {inter:.3f}")
    print(f"Youden threshold (fixed now, from training OOF only): {thr:.4f} "
          f"(sens {sens:.3f}, spec {spec:.3f})")
    print(f"Complete-case refit trained on {int(cc.sum())} patients")
    print("Multi-View OOF AUC (training):",
          {v: round(roc_auc_score(y, p), 4) for v, p in view_oof.items()},
          "| meta-learner", round(roc_auc_score(y, meta_oof), 4))
    print("View sizes:", {v: len(c) for v, c in VIEWS.items()})
    print("Saved:", f"models_v2/final_{arm}.joblib")

print("\nTest data opened in this cell: NO")

In [ ]:
# ========= NB3 PILOT (one cell): would extra labs improve the model? TRAINING patients only ======
# Only random-split TRAINING patients are read from labevents. Test patients are never touched.
# Decision rule fixed BEFORE running: add the labs to the pipeline only if the cross-validated
# AUC gain (realistic version, 20% rule applied) is >= +0.005; otherwise keep the current model.
import os, json, time
import numpy as np, pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score
from v2_models import build

RAW, P2, M2 = "../data/raw/", "../data/processed_v2/", "../models_v2/"
OUT = P2 + "pilot_labs/"
os.makedirs(OUT, exist_ok=True)
def find_raw(name):
    for r, _, fs in os.walk(RAW):
        if name in fs:
            return os.path.join(r, name)
    raise FileNotFoundError(name)

NEW = {51006: "BUN", 50882: "Bicarbonate", 50868: "AnionGap", 50931: "Glucose",
       50809: "Glucose", 50862: "Albumin", 51237: "INR", 50902: "Chloride", 50806: "Chloride",
       50893: "Calcium", 50960: "Magnesium", 50970: "Phosphate"}
RANGE = {"BUN": (1, 300), "Bicarbonate": (2, 60), "AnionGap": (0, 60), "Glucose": (10, 2000),
         "Albumin": (0.5, 7), "INR": (0.5, 20), "Chloride": (60, 150), "Calcium": (3, 20),
         "Magnesium": (0.3, 10), "Phosphate": (0.2, 20)}

# ---- 1. verify item IDs against the dictionary ----
dl = pd.read_csv(find_raw("d_labitems.csv.gz"), usecols=["itemid", "label", "fluid", "category"])
chk = pd.DataFrame({"itemid": list(NEW), "our_name": list(NEW.values())}).merge(dl, how="left")
print(chk.to_string(index=False))
assert chk.label.notna().all(), "item ID not found"

# ---- 2. training patients of the random arm only ----
co = pd.read_csv(P2 + "cohort_final.csv", usecols=["subject_id", "stay_id", "intime"],
                 parse_dates=["intime"])
S = pd.read_csv(P2 + "splits_v2.csv")
train_ids = set(S.stay_id[S.random_split == "train"])
co = co[co.stay_id.isin(train_ids)]
subs, t0 = set(co.subject_id), co.set_index("subject_id")

t = time.time(); parts = []
for ch in pd.read_csv(find_raw("labevents.csv.gz"), chunksize=2_000_000,
                      usecols=["subject_id", "itemid", "charttime", "valuenum"]):
    ch = ch[ch.itemid.isin(NEW) & ch.subject_id.isin(subs)]
    if len(ch):
        parts.append(ch)
lb = pd.concat(parts, ignore_index=True)
lb["charttime"] = pd.to_datetime(lb.charttime)
lb["hours"] = (lb.charttime - lb.subject_id.map(t0.intime)).dt.total_seconds() / 3600
lb = lb[(lb.hours >= -6) & (lb.hours <= 24) & lb.valuenum.notna()].copy()
lb["lab"] = lb.itemid.map(NEW)
lb["stay_id"] = lb.subject_id.map(t0.stay_id)
lo = lb.lab.map(lambda k: RANGE[k][0]); hi = lb.lab.map(lambda k: RANGE[k][1])
lb = lb[(lb.valuenum >= lo) & (lb.valuenum <= hi)]
print(f"\nlabevents scanned ({(time.time() - t) / 60:.1f} min): {len(lb):,} rows kept")

# ---- 3. simple summaries per patient: first, min, max, mean ----
lb = lb.sort_values("hours")
g = lb.groupby(["stay_id", "lab"]).valuenum
feat = pd.concat({"first": g.first(), "min": g.min(), "max": g.max(), "mean": g.mean()},
                 axis=1).unstack("lab")
feat.columns = [f"{lab}_{s}_new" for s, lab in feat.columns]

# ---- 4. join with the CURRENT training table (25 selected features, already prepared) ----
tr = pd.read_csv(P2 + "prep_random_split_train.csv")
y = tr.mortality_28d.astype(int).values
base_feats = pd.read_csv(P2 + "selected_features_random_split.csv").feature.tolist()
X = tr[["stay_id"] + base_feats].merge(feat, left_on="stay_id", right_index=True, how="left")
X = X.drop(columns="stay_id")
newcols = [c for c in X.columns if c.endswith("_new")]

avail = {lab: X[f"{lab}_max_new"].notna().mean() if f"{lab}_max_new" in X else 0
         for lab in RANGE}
print("\n% of training patients with the lab measured (-6h to 24h):")
print(pd.Series(avail).mul(100).round(1).sort_values(ascending=False).to_string())

uni = {c: max(a, 1 - a) for c in newcols
       for a in [roc_auc_score(y[X[c].notna()], X.loc[X[c].notna(), c])]
       if X[c].notna().sum() > 100}
print("\nStrongest single new features (training AUC, direction ignored):")
print(pd.Series(uni).sort_values(ascending=False).head(12).round(3).to_string())

# ---- 5. cross-validated AUC: current 25 features vs + new labs (TRAINING only) ----
params = json.load(open(M2 + "tuning_random_split.json"))["best_params"]["XGB"]
kept = [c for c in newcols if X[c].isna().mean() <= 0.20]           # same 20% rule
sets = {"current 25 features": base_feats,
        "+ new labs (20% rule applied)": base_feats + kept,
        "+ all new labs (no 20% rule)": base_feats + newcols}
res = {k: [] for k in sets}
for trn, val in RepeatedStratifiedKFold(n_splits=5, n_repeats=3,
                                        random_state=2024).split(X, y):
    for k, cols in sets.items():
        m = build("XGB", params, y_train=y[trn], seed=42).fit(X.iloc[trn][cols], y[trn])
        res[k].append(roc_auc_score(y[val], m.predict_proba(X.iloc[val][cols])[:, 1]))
r = pd.DataFrame(res)
print("\nCross-validated AUC (XGB with tuned settings, 5x3 folds, TRAINING patients only):")
for k in sets:
    d = r[k] - r["current 25 features"]
    print(f"   {k:32s} {r[k].mean():.4f} (sd {r[k].std():.4f}) | gain {d.mean():+.4f}, "
          f"better in {(d > 0).mean():.0%} of folds | {len(sets[k])} features")
gain = (r["+ new labs (20% rule applied)"] - r["current 25 features"]).mean()
print(f"\nNew lab features kept by the 20% rule: {len(kept)} of {len(newcols)}")
print(f"PRE-SET RULE: add labs only if gain >= +0.005 -> "
      f"{'ADD THE LABS' if gain >= 0.005 else 'KEEP CURRENT MODEL'} (gain {gain:+.4f})")
r.to_csv(OUT + "pilot_cv_auc.csv", index=False)
print("Test data opened in this cell: NO")

In [ ]:
%%writefile v2_eval.py
# v2_eval.py - test-set evaluation helpers (NB3 cell 7 and NB4).
#  - delong_test(): paired DeLong test for two correlated AUCs (DeLong 1988; fast algorithm of
#    Sun & Xu 2014)
#  - boot_ci(): stratified bootstrap percentile 95% CI for any metric
#  - threshold_metrics(), net_benefit()
import numpy as np
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss


def _midrank(x):
    order = np.argsort(x)
    xs = x[order]
    n = len(x)
    ranks = np.zeros(n)
    i = 0
    while i < n:
        j = i
        while j < n and xs[j] == xs[i]:
            j += 1
        ranks[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(n)
    out[order] = ranks
    return out


def _delong_components(y, scores):
    pos, neg = scores[:, y == 1], scores[:, y == 0]
    m, n = pos.shape[1], neg.shape[1]
    k = scores.shape[0]
    tx = np.array([_midrank(pos[r]) for r in range(k)])
    ty = np.array([_midrank(neg[r]) for r in range(k)])
    tz = np.array([_midrank(np.r_[pos[r], neg[r]]) for r in range(k)])
    aucs = tz[:, :m].sum(axis=1) / (m * n) - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    return aucs, np.atleast_2d(cov)


def delong_test(y, p1, p2):
    """Two-sided p-value for AUC(p1) == AUC(p2) on the same patients."""
    y = np.asarray(y)
    aucs, cov = _delong_components(y, np.vstack([p1, p2]))
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    z = (aucs[0] - aucs[1]) / np.sqrt(var)
    return float(aucs[0]), float(aucs[1]), float(2 * stats.norm.sf(abs(z)))


def delong_ci(y, p):
    aucs, cov = _delong_components(np.asarray(y), np.atleast_2d(p))
    se = np.sqrt(cov[0, 0])
    return float(aucs[0]), float(aucs[0] - 1.96 * se), float(aucs[0] + 1.96 * se)


def boot_ci(y, p, fn, n_boot=2000, seed=42):
    y, p = np.asarray(y), np.asarray(p)
    rng = np.random.default_rng(seed)
    i1, i0 = np.where(y == 1)[0], np.where(y == 0)[0]
    vals = []
    for _ in range(n_boot):
        idx = np.r_[rng.choice(i1, len(i1)), rng.choice(i0, len(i0))]
        vals.append(fn(y[idx], p[idx]))
    return float(fn(y, p)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def threshold_metrics(y, p, thr):
    y, pred = np.asarray(y), (np.asarray(p) >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    sd = lambda a, b: a / b if b else np.nan
    return {"sensitivity": sd(tp, tp + fn), "specificity": sd(tn, tn + fp),
            "ppv": sd(tp, tp + fp), "npv": sd(tn, tn + fn), "tp": tp, "fp": fp, "tn": tn,
            "fn": fn}


def net_benefit(y, p, thresholds):
    y, p = np.asarray(y), np.asarray(p)
    n, prev = len(y), y.mean()
    rows = []
    for t in thresholds:
        pred = p >= t
        tp = ((pred) & (y == 1)).sum(); fp = ((pred) & (y == 0)).sum()
        rows.append({"threshold": t, "model": tp / n - fp / n * t / (1 - t),
                     "treat_all": prev - (1 - prev) * t / (1 - t), "treat_none": 0.0})
    return rows


AUC = roc_auc_score
AP = average_precision_score
BRIER = brier_score_loss

In [ ]:
# ========= NB3 CELL 7b: THE ONE-TIME TEST EVALUATION (both arms) =========
# PART A (training data only): pre-flight checks + LOCK the risk-tier rules.
# PART B: open the test sets once and evaluate with everything already fixed.
# Nothing is changed after seeing these numbers. Re-running gives identical numbers.
import os, json, hashlib, importlib, warnings, datetime
import numpy as np, pandas as pd, joblib
import v2_eval, v2_final, v2_prep
for m in (v2_eval, v2_final, v2_prep):
    importlib.reload(m)
from v2_eval import (delong_test, delong_ci, boot_ci, threshold_metrics, net_benefit,
                     AUC, AP, BRIER)
from v2_final import calibration_slope_intercept

warnings.filterwarnings("ignore")
P2, M2 = "../data/processed_v2/", "../models_v2/"
ARMS = ["random_split", "temporal_split"]
FP = {"random_split": "0a23991d2383", "temporal_split": "8a4eb81d4c96"}

def fp(ids):
    return hashlib.md5(",".join(map(str, sorted(ids))).encode()).hexdigest()[:12]

# ======================= PART A1: pre-flight checks (no test labels read) =======================
need = [P2 + "model_table_v2.csv", P2 + "cohort_delirium_timing.csv",
        P2 + "tpife_thresholds.csv"]
for arm in ARMS:
    need += [P2 + f"prep_{arm}_test.csv", P2 + f"train_oof_final_{arm}.csv",
             M2 + f"final_{arm}.joblib", M2 + f"prep_{arm}.joblib"]
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f"missing files: {missing}"
T = pd.read_csv(P2 + "model_table_v2.csv").set_index("stay_id")
DT = pd.read_csv(P2 + "cohort_delirium_timing.csv").set_index("stay_id")
for c in ["cancer_type", "sex_male", "age_at_admission"]:
    assert c in T.columns, f"model_table_v2 has no column {c}"
assert "cam_positive_by_24h" in DT.columns, f"delirium file columns: {list(DT.columns)}"
KEYS = ["final_calibrated", "final_uncalibrated", "Stacking", "SOFA", "APACHE-II",
        "SV_noTPIFE", "SV_sensitivityA", "SV_complete_case", "cvap", "view_meta", "views",
        "features", "features_noTPIFE", "features_sensitivityA", "threshold",
        "LR", "DT", "RF", "XGB", "CatBoost", "LightGBM"]
BUNDLE = {}
for arm in ARMS:
    B = joblib.load(M2 + f"final_{arm}.joblib")
    assert not [k for k in KEYS if k not in B], f"{arm}: bundle keys missing"
    assert B["final_model_type"] == "SoftVoting", f"{arm}: wrong final model"
    head_cols = set(pd.read_csv(P2 + f"prep_{arm}_test.csv", nrows=0).columns)
    used = set(B["features"] + B["features_noTPIFE"] + B["features_sensitivityA"]
               + ["sofa_total", "apache_ii_total"]
               + [c for cols in B["views"].values() for c in cols])
    assert not (used - head_cols), f"{arm}: test file lacks {sorted(used - head_cols)[:5]}"
    ids = pd.read_csv(P2 + f"prep_{arm}_test.csv", usecols=["stay_id"]).stay_id
    assert fp(ids) == FP[arm], f"{arm}: test patients changed - stop"
    assert ids.isin(T.index).all() and ids.isin(DT.index).all(), f"{arm}: ids not found"
    BUNDLE[arm] = B
print("Pre-flight checks: all files, columns, model bundles and test fingerprints OK")

# ============ PART A2: LOCK risk tiers from TRAINING out-of-fold predictions only ============
# Rule (fixed here, before the test set is opened):
#  - 3 risk tiers = tertiles of the out-of-fold calibrated training predictions
#  - a patient is 'uncertain' if their Venn-Abers interval [p0, p1] crosses a tier cut-off
#    (the model cannot say which tier they belong to); the same idea is applied to the
#    Youden decision threshold
LOCK = {}
for arm in ARMS:
    oof = pd.read_csv(P2 + f"train_oof_final_{arm}.csv")
    q1, q2 = np.quantile(oof.oof_calibrated, [1 / 3, 2 / 3])
    assert q1 < q2, f"{arm}: tier cut-offs collapsed"
    tier = np.select([oof.oof_calibrated < q1, oof.oof_calibrated < q2],
                     ["Low", "Intermediate"], "High")
    tr_tab = oof.assign(tier=tier).groupby("tier").y.agg(["size", "mean"])
    LOCK[arm] = {"tier_cut_low": float(q1), "tier_cut_high": float(q2),
                 "threshold": float(BUNDLE[arm]["threshold"]),
                 "tier_rule": "tertiles of training OOF calibrated risk",
                 "uncertain_rule": "Venn-Abers [p0,p1] crosses a tier cut-off"}
    path = M2 + f"locked_rules_{arm}.json"
    if os.path.exists(path):                                 # re-run must give the same rules
        old = json.load(open(path))
        assert all(np.isclose(old[k], LOCK[arm][k]) for k in
                   ["tier_cut_low", "tier_cut_high", "threshold"]), "locked rules changed"
    else:
        json.dump(LOCK[arm], open(path, "w"), indent=2)
    print(f"{arm}: tiers Low < {q1:.3f} <= Intermediate < {q2:.3f} <= High | training "
          f"mortality by tier {tr_tab['mean'].round(3).to_dict()} | n {tr_tab['size'].to_dict()}")

def tiers(p, L):
    return np.select([p < L["tier_cut_low"], p < L["tier_cut_high"]],
                     ["Low", "Intermediate"], "High")

def age_group(a):
    return np.select([a < 60, a < 75], ["<60", "60-74"], ">=75")

log = M2 + "test_opened_log.json"
if not os.path.exists(log):
    json.dump({"first_opened": datetime.datetime.now().isoformat(timespec="seconds")},
              open(log, "w"))
print("Test sets opened at:", json.load(open(log))["first_opened"], "\n")

# =============================== PART B: test evaluation ===============================
all_rows, sub_rows, tier_rows = [], [], []
for arm in ARMS:
    B, L = BUNDLE[arm], LOCK[arm]
    te = pd.read_csv(P2 + f"prep_{arm}_test.csv")
    y = te.pop("mortality_28d").astype(int).values
    sid = te.pop("stay_id").values
    f, thr = B["features"], B["threshold"]

    # ---------------- predictions ----------------
    P = {"SoftVoting_calibrated": B["final_calibrated"].predict_proba(te[f])[:, 1],
         "SoftVoting": B["final_uncalibrated"].predict_proba(te[f])[:, 1],
         "Stacking": B["Stacking"].predict_proba(te[f])[:, 1],
         "SOFA": B["SOFA"].predict_proba(te[["sofa_total"]])[:, 1],
         "APACHE-II": B["APACHE-II"].predict_proba(te[["apache_ii_total"]])[:, 1],
         "SV_noTPIFE": B["SV_noTPIFE"].predict_proba(te[B["features_noTPIFE"]])[:, 1],
         "SV_with_measurement": B["SV_sensitivityA"].predict_proba(
             te[B["features_sensitivityA"]])[:, 1]}
    for n in ["LR", "DT", "RF", "XGB", "CatBoost", "LightGBM"]:
        P[n] = B[n].predict_proba(te[f])[:, 1]
    views = {v: B[f"view_{v}"].predict_proba(te[c])[:, 1] for v, c in B["views"].items()}
    P["MultiView_meta"] = B["view_meta"].predict_proba(
        np.column_stack(list(views.values())))[:, 1]
    va = B["cvap"].predict(te[f])
    pc = P["SoftVoting_calibrated"]

    # ---------------- headline metrics (final calibrated model) ----------------
    auc, lo, hi = boot_ci(y, pc, AUC)
    ap, aplo, aphi = boot_ci(y, pc, AP)
    br, brlo, brhi = boot_ci(y, pc, BRIER)
    slope, inter = calibration_slope_intercept(y, pc)
    tm = threshold_metrics(y, pc, thr)
    head = {"arm": arm, "n": len(y), "deaths": int(y.sum()), "auc": auc, "auc_lo": lo,
            "auc_hi": hi, "auprc": ap, "auprc_lo": aplo, "auprc_hi": aphi, "brier": br,
            "brier_lo": brlo, "brier_hi": brhi, "cal_slope": slope, "cal_intercept": inter,
            "observed_mortality": y.mean(), "mean_predicted": pc.mean(),
            "expected_observed_ratio": pc.mean() / y.mean(), "threshold": thr, **tm}
    for k in ["sensitivity", "specificity", "ppv", "npv"]:
        fn = (lambda kk: lambda yy, pp: threshold_metrics(yy, pp, thr)[kk])(k)
        _, head[k + "_lo"], head[k + "_hi"] = boot_ci(y, pc, fn, n_boot=1000)
    all_rows.append(head)

    # ---------------- every model: AUC (DeLong CI) ----------------
    comp = []
    for name, p in P.items():
        a, l, h = delong_ci(y, p)
        comp.append({"arm": arm, "model": name, "auc": a, "lo": l, "hi": h,
                     "brier": BRIER(y, p) if name in ("SoftVoting_calibrated", "SOFA",
                                                      "APACHE-II") else np.nan})
    for v, p in views.items():
        a, l, h = delong_ci(y, p)
        comp.append({"arm": arm, "model": f"View_{v}", "auc": a, "lo": l, "hi": h})
    a, l, h = delong_ci(y, va["p"])
    comp.append({"arm": arm, "model": "SoftVoting_VennAbers", "auc": a, "lo": l, "hi": h,
                 "brier": BRIER(y, va["p"])})
    comp = pd.DataFrame(comp)

    # ---------------- pre-specified DeLong tests (uncalibrated scores) ----------------
    dl = []
    for other, why in [("Stacking", "CV runner-up"), ("SOFA", "baseline"),
                       ("APACHE-II", "baseline"), ("SV_noTPIFE", "TPIFE contribution"),
                       ("SV_with_measurement", "measurement sensitivity")]:
        a1, a2, pv = delong_test(y, P["SoftVoting"], P[other])
        dl.append({"arm": arm, "comparison": f"SoftVoting vs {other}", "purpose": why,
                   "auc_softvoting": a1, "auc_other": a2, "diff": a1 - a2, "p_delong": pv})
    dl = pd.DataFrame(dl)

    # ---------------- complete-case refit (on complete-case test patients) ----------------
    prep = joblib.load(M2 + f"prep_{arm}.joblib")
    drop = ["subject_id", "hadm_id", "mortality_28d", "est_calendar_year", "temporal_set"]
    raw = v2_prep.add_fixed_features(T.loc[sid, [c for c in T.columns if c not in drop]])
    cc = prep.complete_case_mask(raw).values
    cc_row = {"arm": arm, "n_complete_case": int(cc.sum()),
              "auc_complete_case_model": np.nan, "auc_main_model_same_patients": np.nan}
    if cc.sum() >= 30 and y[cc].min() != y[cc].max():
        p_cc = B["SV_complete_case"].predict_proba(te.loc[cc, f])[:, 1]
        cc_row["auc_complete_case_model"] = AUC(y[cc], p_cc)
        cc_row["auc_main_model_same_patients"] = AUC(y[cc], pc[cc])

    # ---------------- risk tiers + uncertainty (rules locked in Part A) ----------------
    tr = tiers(pc, L)
    cuts = [L["tier_cut_low"], L["tier_cut_high"]]
    unc_tier = np.zeros(len(y), bool)
    for c in cuts:
        unc_tier |= (va["p0"] < c) & (va["p1"] >= c)
    unc_thr = (va["p0"] < thr) & (va["p1"] >= thr)
    for lev in ["Low", "Intermediate", "High"]:
        m = tr == lev
        tier_rows.append({"arm": arm, "tier": lev, "n": int(m.sum()),
                          "deaths": int(y[m].sum()),
                          "obs_mortality": y[m].mean() if m.any() else np.nan,
                          "mean_predicted": pc[m].mean() if m.any() else np.nan,
                          "pct_uncertain": unc_tier[m].mean() if m.any() else np.nan,
                          "obs_mortality_certain": y[m & ~unc_tier].mean()
                          if (m & ~unc_tier).any() else np.nan,
                          "obs_mortality_uncertain": y[m & unc_tier].mean()
                          if (m & unc_tier).any() else np.nan})
    acc = lambda m: ((pc[m] >= thr) == y[m]).mean() if m.any() else np.nan
    unc_row = {"pct_uncertain_at_threshold": unc_thr.mean(),
               "accuracy_certain": acc(~unc_thr), "accuracy_uncertain": acc(unc_thr)}

    # ---------------- subgroups (fairness / consistency) ----------------
    g = pd.DataFrame({"y": y, "p": pc,
                      "cancer_type": T.loc[sid, "cancer_type"].values,
                      "sex": np.where(T.loc[sid, "sex_male"].values == 1, "male", "female"),
                      "age_group": age_group(T.loc[sid, "age_at_admission"].values),
                      "delirium_by_24h": np.where(
                          DT.loc[sid, "cam_positive_by_24h"].values == 1,
                          "CAM-ICU+ by 24h", "not by 24h")})
    for var in ["cancer_type", "sex", "age_group", "delirium_by_24h"]:
        for lev, d in g.groupby(var):
            r = {"arm": arm, "subgroup": var, "level": lev, "n": len(d),
                 "deaths": int(d.y.sum()), "obs_mortality": d.y.mean(),
                 "mean_predicted": d.p.mean(), "brier": BRIER(d.y, d.p)}
            if d.y.nunique() == 2 and d.y.sum() >= 10 and (1 - d.y).sum() >= 10:
                r["auc"], r["auc_lo"], r["auc_hi"] = boot_ci(d.y.values, d.p.values, AUC,
                                                             n_boot=1000)
                r.update({k: v for k, v in threshold_metrics(d.y, d.p, thr).items()
                          if k in ("sensitivity", "specificity")})
            r["reliable"] = bool(d.y.sum() >= 30 and (1 - d.y).sum() >= 30)
            sub_rows.append(r)

    # ---------------- save patient-level predictions for NB4 ----------------
    out = pd.DataFrame({"stay_id": sid, "y": y, **{f"p_{k}": v for k, v in P.items()},
                        **{f"p_View_{k}": v for k, v in views.items()},
                        "va_p": va["p"], "va_p0": va["p0"], "va_p1": va["p1"],
                        "va_width": va["width"], "tier": tr,
                        "uncertain_tier": unc_tier, "uncertain_threshold": unc_thr})
    out.to_csv(P2 + f"test_predictions_{arm}.csv", index=False)
    comp.to_csv(P2 + f"test_model_auc_{arm}.csv", index=False)
    dl.to_csv(P2 + f"test_delong_{arm}.csv", index=False)
    pd.DataFrame(net_benefit(y, pc, np.round(np.arange(0.05, 0.81, 0.05), 2))).to_csv(
        P2 + f"test_net_benefit_{arm}.csv", index=False)
    pd.DataFrame([{**cc_row, **unc_row}]).to_csv(P2 + f"test_complete_case_{arm}.csv",
                                                 index=False)

    # ---------------- print ----------------
    print("=" * 74, f"\n{arm.upper()} TEST SET: n={len(y)}, deaths={int(y.sum())} "
          f"({y.mean():.1%}) | fingerprint OK\n" + "=" * 74)
    print(f"FINAL Soft-Voting (calibrated): AUC {auc:.3f} ({lo:.3f}-{hi:.3f}) | "
          f"AUPRC {ap:.3f} ({aplo:.3f}-{aphi:.3f}) | Brier {br:.3f} ({brlo:.3f}-{brhi:.3f})")
    print(f"Calibration slope {slope:.3f}, intercept {inter:.3f} | mean predicted "
          f"{pc.mean():.3f} vs observed {y.mean():.3f} (E/O {pc.mean() / y.mean():.3f})")
    print(f"At threshold {thr:.3f}: sens {head['sensitivity']:.3f} "
          f"({head['sensitivity_lo']:.3f}-{head['sensitivity_hi']:.3f}), spec "
          f"{head['specificity']:.3f} ({head['specificity_lo']:.3f}-"
          f"{head['specificity_hi']:.3f}), PPV {head['ppv']:.3f}, NPV {head['npv']:.3f}")
    print(f"Venn-Abers: AUC {AUC(y, va['p']):.3f}, Brier {BRIER(y, va['p']):.3f}, "
          f"width median {np.median(va['width']):.3f} (IQR "
          f"{np.percentile(va['width'], 25):.3f}-{np.percentile(va['width'], 75):.3f})")
    print(f"Uncertain at threshold: {unc_thr.mean():.1%} of patients | accuracy certain "
          f"{unc_row['accuracy_certain']:.3f} vs uncertain {unc_row['accuracy_uncertain']:.3f}")
    print("\nAll models (AUC, DeLong 95% CI):")
    print(comp.sort_values("auc", ascending=False).round(3).to_string(index=False))
    print("\nPre-specified DeLong tests:")
    print(dl.round(4).to_string(index=False))
    print(f"\nComplete-case: n={cc_row['n_complete_case']} | refit model AUC "
          f"{cc_row['auc_complete_case_model']:.3f} vs main model on same patients "
          f"{cc_row['auc_main_model_same_patients']:.3f}")
    print("\nRisk tiers (locked rules):")
    print(pd.DataFrame([r for r in tier_rows if r["arm"] == arm]).drop(columns="arm")
          .round(3).to_string(index=False))
    print()

pd.DataFrame(all_rows).to_csv(P2 + "test_headline_metrics.csv", index=False)
pd.DataFrame(tier_rows).to_csv(P2 + "test_risk_tiers.csv", index=False)
S = pd.DataFrame(sub_rows)
S.to_csv(P2 + "test_subgroups.csv", index=False)
print("=" * 74 + "\nSUBGROUPS (calibrated final model; 'reliable' = >=30 deaths and "
      ">=30 survivors)\n" + "=" * 74)
cols = ["arm", "subgroup", "level", "n", "deaths", "auc", "auc_lo", "auc_hi",
        "obs_mortality", "mean_predicted", "sensitivity", "specificity", "reliable"]
print(S[[c for c in cols if c in S]].round(3).to_string(index=False))
print("\nSaved: test_headline_metrics, test_model_auc_<arm>, test_delong_<arm>, "
      "test_subgroups, test_risk_tiers, test_predictions_<arm>, test_net_benefit_<arm>, "
      "test_complete_case_<arm> (.csv) + models_v2/locked_rules_<arm>.json")

In [ ]:
# ========= NB3 VERIFICATION CELL: checks the WHOLE of Notebook 3 against the methodology =========
# Read-only: re-computes key numbers from the saved files and compares. Nothing is re-trained,
# nothing is overwritten. PASS = correct, FAIL = must be looked at, INFO = number for the paper.
import os, json, hashlib, importlib, warnings, datetime, glob
import numpy as np, pandas as pd, joblib
from scipy import stats
from sklearn.metrics import roc_auc_score, brier_score_loss
import v2_prep, v2_select, v2_models, v2_final, v2_eval
for m in (v2_prep, v2_select, v2_models, v2_final, v2_eval):
    importlib.reload(m)
from v2_prep import load_table, add_fixed_features, one_hot, PROTECTED, LABEL
from v2_select import is_measurement, feature_family, TPIFE_FAMILIES
from v2_models import MODELS, at_boundary
from v2_final import youden_threshold
from v2_eval import delong_test

warnings.filterwarnings("ignore")
P2, M2, CV = "../data/processed_v2/", "../models_v2/", "../data/processed_v2/cv/"
ARMS = ["random_split", "temporal_split"]
FP = {"random_split": "0a23991d2383", "temporal_split": "8a4eb81d4c96"}
SIZES = {"random_split": (2221, 556), "temporal_split": (1373, 1404)}
EXP = {"random_split": {"B": 25, "A": 27, "C": 24, "cc": 1402},
       "temporal_split": {"B": 32, "A": 50, "C": 22, "cc": 874}}
R = []

def check(section, name, ok, detail=""):
    R.append({"section": section, "check": name,
              "result": "PASS" if ok else "FAIL", "detail": str(detail)})

def info(section, name, detail):
    R.append({"section": section, "check": name, "result": "INFO", "detail": str(detail)})

def fp(ids):
    return hashlib.md5(",".join(map(str, sorted(ids))).encode()).hexdigest()[:12]

def same(a, b, tol=1e-9):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return a.shape == b.shape and np.allclose(a, b, rtol=tol, atol=tol, equal_nan=True)

T, S, FEATURES = load_table()
X_all = add_fixed_features(T[FEATURES])
y_all = T[LABEL].astype(int).values

# ---------------------------------- 1. SPLITS (cell 1) ----------------------------------
s = "1 splits"
check(s, "2,777 patients, split file aligned with table", len(T) == 2777
      and (T.stay_id.values == S.stay_id.values).all(), len(T))
check(s, "one ICU stay per patient (subject_id unique)", T.subject_id.is_unique)
WORDS = ["mortality", "death", "expire", "discharge", "outtime", "subtype", "anchor",
         "temporal", "immortal", "hospice", "calendar", "year", "delirium", "cam_"]
TOK = ["dod", "los", "died", "dead", "label", "target", "outcome", "id"]
bad = [c for c in FEATURES if any(w in c.lower() for w in WORDS)
       or any(t in c.lower().split("_") for t in TOK)]
check(s, "no leaky / ID / outcome columns among 391 features",
      not bad and len(FEATURES) == 391, bad[:5] if bad else len(FEATURES))
for arm in ARMS:
    tr, te = (S[arm] == "train").values, (S[arm] == "test").values
    check(s, f"{arm}: train/test disjoint and cover everyone",
          not (tr & te).any() and (tr | te).all())
    check(s, f"{arm}: sizes {SIZES[arm]}", (tr.sum(), te.sum()) == SIZES[arm],
          (int(tr.sum()), int(te.sum())))
    check(s, f"{arm}: test fingerprint unchanged", fp(S.stay_id[te]) == FP[arm],
          fp(S.stay_id[te]))
    check(s, f"{arm}: no patient (subject) in both parts",
          not set(S.subject_id[tr]) & set(S.subject_id[te]))
    d = abs(y_all[tr].mean() - y_all[te].mean())
    check(s, f"{arm}: mortality similar in train and test (<1.5 points)", d < 0.015,
          f"{y_all[tr].mean():.3f} vs {y_all[te].mean():.3f}")
tr, te = (S.temporal_split == "train").values, (S.temporal_split == "test").values
check(s, "era split: train <= 2017, test > 2017",
      (S.est_calendar_year[tr] <= 2017).all() and (S.est_calendar_year[te] > 2017).all())

# ------------------------------ 2. PREPARATION (cell 2) ------------------------------
s = "2 preparation"
PREP = {}
for arm in ARMS:
    tr = (S[arm] == "train").values
    prep = joblib.load(M2 + f"prep_{arm}.joblib"); PREP[arm] = prep
    ftr = pd.read_csv(P2 + f"prep_{arm}_train.csv")
    fte = pd.read_csv(P2 + f"prep_{arm}_test.csv")
    check(s, f"{arm}: prepared train/test hold exactly the split patients",
          (ftr.stay_id.values == T.stay_id[tr].values).all()
          and (fte.stay_id.values == T.stay_id[~tr].values).all())
    check(s, f"{arm}: outcome in prepared files matches the table",
          (ftr[LABEL].values == y_all[tr]).all() and (fte[LABEL].values == y_all[~tr]).all())
    check(s, f"{arm}: no missing values after preparation",
          not ftr.isna().any().any() and not fte.isna().any().any())
    check(s, f"{arm}: same columns in train, test and fitted preparation",
          list(ftr.columns[2:]) == list(fte.columns[2:]) == prep.out_cols)
    check(s, f"{arm}: outcome not used as an input", LABEL not in prep.out_cols)
    raw_tr = one_hot(X_all[tr])
    miss = raw_tr.isna().mean()
    check(s, f"{arm}: missing-% rule learned from TRAINING rows only",
          same(miss.sort_index(), prep.train_missing.sort_index(), 1e-12))
    kept = [c for c in prep.cols]
    lim = pd.Series(0.20, index=kept)
    lim[[c for c in PROTECTED if c in lim.index]] = 0.30
    check(s, f"{arm}: every kept feature <=20% missing in train (lactate <=30%)",
          (miss[kept] <= lim + 1e-12).all())
    check(s, f"{arm}: lactate kept (pre-specified exception)",
          "Lactate_max" in prep.out_cols and "Lactate_measured" in prep.out_cols)
    Xo = raw_tr.drop(columns=prep.dropped_missing + prep.dropped_constant
                     + prep.dropped_duplicate)
    check(s, f"{arm}: clipping limits are TRAINING 0.5/99.5% quantiles",
          same(Xo[prep.cont].quantile(0.005), prep.lo, 1e-9)
          and same(Xo[prep.cont].quantile(0.995), prep.hi, 1e-9))
    re_te = prep.transform(X_all[~tr])
    check(s, f"{arm}: test file = training-fitted preparation applied to test",
          same(re_te.values, fte[prep.out_cols].values, 1e-7))
    info(s, f"{arm}: model inputs after preparation", len(prep.out_cols))

# ---------------------------- 3. FEATURE SELECTION (cell 3) ----------------------------
s = "3 selection"
dec = json.load(open(M2 + "feature_pool_decision.json"))
check(s, "pool decided once (random arm) and used in both arms",
      dec["random_split"]["pool_used"] == dec["temporal_split"]["pool_used"]
      == "B_no_measurement")
ra, rb = dec["random_split"]["A_all"]["auc"], dec["random_split"]["B_no_measurement"]["auc"]
check(s, "pool rule: B used because AUC_B >= AUC_A - 0.005", rb >= ra - 0.005,
      f"A {ra} vs B {rb}")
SEL = {}
for arm in ARMS:
    out_cols = PREP[arm].out_cols
    for pool, suf, key in [("B_no_measurement", "", "B"), ("A_all", "_sensitivityA", "A"),
                           ("C_no_TPIFE", "_noTPIFE", "C")]:
        v = pd.read_csv(P2 + f"selection_votes_{arm}_{pool}.csv")
        f = pd.read_csv(P2 + f"selected_features_{arm}{suf}.csv").feature.tolist()
        SEL[(arm, key)] = f
        check(s, f"{arm} pool {key}: features = chosen in >=3 of 5 stability runs",
              sorted(f) == sorted(v.feature[v.votes >= 3]))
        check(s, f"{arm} pool {key}: {EXP[arm][key]} features", len(f) == EXP[arm][key],
              len(f))
        check(s, f"{arm} pool {key}: all features exist in the prepared data",
              set(f) <= set(out_cols))
    check(s, f"{arm}: primary set has no measurement-process features",
          not any(is_measurement(c) for c in SEL[(arm, "B")]))
    check(s, f"{arm}: no-TPIFE set has no TPIFE/CSI/stability/OSBI features",
          not any(feature_family(c) in TPIFE_FAMILIES for c in SEL[(arm, "C")]))
    leak = [c for c in SEL[(arm, "B")] + SEL[(arm, "A")]
            if any(w in c.lower() for w in WORDS)]
    check(s, f"{arm}: no leaky names among selected features", not leak, leak)

# ------------------------------------ 4. TUNING (cell 4) ------------------------------------
s = "4 tuning"
NATURAL = {"min_samples_leaf", "min_child_weight"}          # value 1 = natural lower limit
TUNE = {}
for arm in ARMS:
    tj = json.load(open(M2 + f"tuning_{arm}.json")); TUNE[arm] = tj["best_params"]
    check(s, f"{arm}: all 6 models tuned", set(MODELS) <= set(tj["best_params"]))
    check(s, f"{arm}: class weights kept (SMOTEENN not better by >0.005)",
          tj["imbalance_method"] == "class_weights", tj.get("imbalance_check", ""))
    if "features" in tj:
        check(s, f"{arm}: tuned on the final primary features",
              sorted(tj["features"]) == sorted(SEL[(arm, "B")]))
    edges = {n: at_boundary(n, tj["best_params"][n]) for n in MODELS}
    odd = {n: [e for e in v if e.split("=")[0] not in NATURAL] for n, v in edges.items()}
    odd = {n: v for n, v in odd.items() if v}
    check(s, f"{arm}: no tuned value stuck at a range edge (except natural limits)",
          not odd, odd if odd else "only natural limits")

# ------------------------------ 5. NESTED CV (cell 5) ------------------------------
s = "5 nested CV"
oof = pd.read_csv(CV + "cv_oof_predictions.csv")
fm = pd.read_csv(CV + "cv_fold_metrics.csv")
summ = pd.read_csv(CV + "cv_summary.csv")
train_ids = set(S.stay_id[S.random_split == "train"])
test_ids = set(S.stay_id[S.random_split == "test"])
check(s, "NO test patient anywhere in nested-CV predictions",
      not set(oof.stay_id) & test_ids)
check(s, "all random-arm training patients used", set(oof.stay_id) == train_ids)
cnt = oof.groupby(["pool", "model", "repeat"]).stay_id.agg(["size", "nunique"])
check(s, "each patient predicted exactly once per repeat, model and pool",
      ((cnt["size"] == len(train_ids)) & (cnt["nunique"] == len(train_ids))).all())
nf = fm.groupby(["pool", "model"]).size()
check(s, "25 outer folds for every model and pool", (nf == 25).all(), nf.unique())
re_auc = oof.groupby(["repeat", "fold", "pool", "model"]).apply(
    lambda d: roc_auc_score(d.y, d.prob)).rename("re").reset_index()
mm = fm.merge(re_auc, on=["repeat", "fold", "pool", "model"])
check(s, "fold AUCs re-computed from saved predictions match",
      len(mm) == len(fm) and same(mm.auc, mm.re, 1e-9))
ms = fm.groupby(["pool", "model"]).auc.mean().rename("re").reset_index().merge(
    summ, on=["pool", "model"])
check(s, "CV summary = mean of the 25 fold AUCs", same(ms.auc_mean, ms.re, 1e-4))
c = summ.query("pool == 'B_primary'").set_index("model")
CAND = ["LR", "RF", "XGB", "CatBoost", "LightGBM", "SoftVoting", "Stacking"]
RANK = {"LR": 0, "RF": 1, "XGB": 1, "CatBoost": 1, "LightGBM": 1, "SoftVoting": 2,
        "Stacking": 3}
c = c.loc[CAND]
near = c[c.auc_mean >= c.auc_mean.max() - 0.005].copy()
near["rank"] = near.index.map(RANK)
near = near.sort_values(["rank", "auc_mean", "brier_mean"], ascending=[True, False, True])
check(s, "written model-choice rule re-applied -> SoftVoting", near.index[0] == "SoftVoting",
      near.index[0])
info(s, "nested-CV AUC (primary pool)", c.auc_mean.round(4).to_dict())
piv = fm.pivot_table(index=["repeat", "fold"], columns=["pool", "model"], values="auc")
def ctest(d):
    t = d.mean() / np.sqrt((1 / len(d) + 1 / 4) * d.var(ddof=1))
    return 2 * stats.t.sf(abs(t), df=len(d) - 1)
saved_t = pd.read_csv(CV + "cv_corrected_ttests.csv").set_index("comparison").p_value
p_re = ctest(piv[("B_primary", "SoftVoting")] - piv[("C_no_TPIFE", "SoftVoting")])
check(s, "corrected t-test (TPIFE) re-computed matches",
      abs(p_re - saved_t["SoftVoting: primary vs no-TPIFE"]) < 1e-4, round(p_re, 4))

# ------------------------------ 6. FINAL MODELS (cell 6) ------------------------------
s = "6 final model"
BUN = {}
for arm in ARMS:
    B = joblib.load(M2 + f"final_{arm}.joblib"); BUN[arm] = B
    tr = (S[arm] == "train").values
    check(s, f"{arm}: final model type SoftVoting (same in both arms)",
          B["final_model_type"] == "SoftVoting")
    check(s, f"{arm}: model uses exactly the selected primary features",
          B["features"] == SEL[(arm, "B")])
    check(s, f"{arm}: ablation models use the saved no-TPIFE / pool-A sets",
          B["features_noTPIFE"] == SEL[(arm, "C")]
          and B["features_sensitivityA"] == SEL[(arm, "A")])
    check(s, f"{arm}: model uses the tuned settings", B["params"] == TUNE[arm])
    cal = B["final_calibrated"]
    check(s, f"{arm}: isotonic calibration, cv=5, one refit (ensemble=False)",
          cal.method == "isotonic" and cal.cv == 5 and cal.ensemble is False)
    ot = pd.read_csv(P2 + f"train_oof_final_{arm}.csv")
    check(s, f"{arm}: threshold source = training patients only (no test ids)",
          (ot.stay_id.values == T.stay_id[tr].values).all() and (ot.y.values == y_all[tr]).all())
    thr, _, _ = youden_threshold(ot.y.values, ot.oof_calibrated.values)
    check(s, f"{arm}: Youden threshold re-computed matches", abs(thr - B["threshold"]) < 1e-12,
          round(B["threshold"], 4))
    check(s, f"{arm}: complete-case refit trained on {EXP[arm]['cc']} patients",
          B["train_complete_case_n"] == EXP[arm]["cc"], B["train_complete_case_n"])
    Xtr = pd.read_csv(P2 + f"prep_{arm}_train.csv")
    va = B["cvap"].predict(Xtr[B["features"]].iloc[:40])
    check(s, f"{arm}: Venn-Abers lower <= upper bound (40 patients)",
          (va["p0"] <= va["p1"] + 1e-12).all())

# ------------------------------ 7. TEST EVALUATION (cell 7) ------------------------------
s = "7 test"
log = json.load(open(M2 + "test_opened_log.json"))
opened = datetime.datetime.fromisoformat(log["first_opened"]) + datetime.timedelta(seconds=1)
info(s, "test sets first opened", log["first_opened"])
frozen = ([P2 + "model_table_v2.csv", P2 + "splits_v2.csv",
           M2 + "feature_pool_decision.json", P2 + "pilot_labs/pilot_cv_auc.csv"]
          + glob.glob(CV + "*.csv") + glob.glob(P2 + "selected_features_*.csv")
          + glob.glob(P2 + "selection_votes_*.csv")
          + [p for a in ARMS for p in [M2 + f"final_{a}.joblib", M2 + f"prep_{a}.joblib",
                                      M2 + f"tuning_{a}.json", M2 + f"locked_rules_{a}.json",
                                      P2 + f"prep_{a}_train.csv", P2 + f"prep_{a}_test.csv",
                                      P2 + f"train_oof_final_{a}.csv"]])
late = [os.path.basename(f) for f in frozen if os.path.exists(f) and
        datetime.datetime.fromtimestamp(os.path.getmtime(f)) > opened]
check(s, "nothing in the pipeline was changed after the test was opened", not late, late)
pil = pd.read_csv(P2 + "pilot_labs/pilot_cv_auc.csv")
g = (pil.iloc[:, 1] - pil.iloc[:, 0]).mean()
check(s, "labs pilot: gain < +0.005 -> current model kept (25 features)",
      g < 0.005 and len(BUN["random_split"]["features"]) == 25, f"gain {g:+.4f}")
H = pd.read_csv(P2 + "test_headline_metrics.csv").set_index("arm")
DLs = []
for arm in ARMS:
    B, te_m = BUN[arm], (S[arm] == "test").values
    L = json.load(open(M2 + f"locked_rules_{arm}.json"))
    ot = pd.read_csv(P2 + f"train_oof_final_{arm}.csv")
    q = np.quantile(ot.oof_calibrated, [1 / 3, 2 / 3])
    check(s, f"{arm}: risk tiers = tertiles of TRAINING predictions",
          same(q, [L["tier_cut_low"], L["tier_cut_high"]], 1e-12), np.round(q, 3))
    pr = pd.read_csv(P2 + f"test_predictions_{arm}.csv")
    check(s, f"{arm}: test predictions = exactly the test patients, correct outcome",
          (pr.stay_id.values == T.stay_id[te_m].values).all() and (pr.y.values ==
                                                                   y_all[te_m]).all())
    fte = pd.read_csv(P2 + f"prep_{arm}_test.csv")
    check(s, f"{arm}: saved test predictions reproduced by the saved final model",
          same(B["final_calibrated"].predict_proba(fte[B["features"]])[:, 1],
               pr.p_SoftVoting_calibrated, 1e-7)
          and same(B["final_uncalibrated"].predict_proba(fte[B["features"]])[:, 1],
                   pr.p_SoftVoting, 1e-7))
    check(s, f"{arm}: headline AUC / Brier re-computed match",
          abs(roc_auc_score(pr.y, pr.p_SoftVoting_calibrated) - H.loc[arm, "auc"]) < 1e-9
          and abs(brier_score_loss(pr.y, pr.p_SoftVoting_calibrated) - H.loc[arm, "brier"])
          < 1e-9, round(H.loc[arm, "auc"], 4))
    dl = pd.read_csv(P2 + f"test_delong_{arm}.csv")
    rep = [delong_test(pr.y.values, pr.p_SoftVoting.values,
                       pr["p_" + r.comparison.split(" vs ")[1]].values)[2]
           for r in dl.itertuples()]
    check(s, f"{arm}: pre-specified DeLong p-values re-computed match",
          same(rep, dl.p_delong, 1e-9))
    tier = np.select([pr.p_SoftVoting_calibrated < L["tier_cut_low"],
                      pr.p_SoftVoting_calibrated < L["tier_cut_high"]],
                     ["Low", "Intermediate"], "High")
    check(s, f"{arm}: every test patient in the tier given by the locked rule",
          (tier == pr.tier.values).all())
    check(s, f"{arm}: Venn-Abers lower <= upper bound for all test patients",
          (pr.va_p0 <= pr.va_p1 + 1e-12).all())
sg = pd.read_csv(P2 + "test_subgroups.csv")
n_ok = all(sg[(sg.arm == a) & (sg.subgroup == v)].n.sum() == SIZES[a][1]
           for a in ARMS for v in sg.subgroup.unique())
check(s, "subgroups cover every test patient exactly once", n_ok)

# ------------------------------------ REPORT ------------------------------------
R = pd.DataFrame(R)
pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 200)
for sec, d in R.groupby("section", sort=False):
    print(f"\n=== {sec.upper()} ===")
    for r in d.itertuples():
        print(f"  [{r.result}] {r.check}" + (f"  ->  {r.detail}" if r.detail and
                                              (r.result != "PASS") else ""))
n_pass, n_fail = (R.result == "PASS").sum(), (R.result == "FAIL").sum()
R.to_csv(P2 + "nb3_verification.csv", index=False)
print(f"\nTOTAL: {n_pass} PASS, {n_fail} FAIL, {(R.result == 'INFO').sum()} INFO "
      "-> saved processed_v2/nb3_verification.csv")
print("ALL NB3 CHECKS PASSED" if n_fail == 0 else "SOME CHECKS FAILED - send me this output")